# SNU-KG: GraphRAG + kg-gen 통합 파이프라인

이 노트북은 Microsoft의 GraphRAG와 kg-gen을 통합하여 한국 농업 지식그래프를 생성하는 파이프라인입니다.

### 주요 특징:
- **언어 자동 통합**: kg-gen 클러스터링으로 "토마토"와 "TOMATO" 같은 다른 언어 엔티티 자동 통합
- **중복 제거**: kg-gen의 K-means + LLM 기반 중복 제거
- **계층적 구조**: GraphRAG의 Leiden 커뮤니티 탐지
- **임베딩 최적화**: description 제외, 핵심 정보만 활용

## 0. 실행 전 확인사항

1. **환경 변수 설정**:
   ```bash
   export OPENAI_API_KEY="your-api-key"
   ```

2. **필요한 패키지 설치**:
   ```bash
   # GraphRAG 설치
   pip install graphrag
   
   # kg-gen 설치
   cd kg_gen/kg-gen
   pip install -e '.[dev]'
   
   # 추가 패키지
   pip install sentence-transformers scikit-learn pyvis
   ```

3. **데이터 파일 확인**:
   - `<PROJECT_ROOT>/rag_dataset/processed_texts/`

4. **메모리 요구사항**: 최소 8GB RAM

## 1. 환경 설정 및 라이브러리 임포트

In [ ]:
# 필수 라이브러리 임포트
import os
import sys
import json
import yaml
import subprocess
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import time
import shutil
from typing import Dict, List, Set, Tuple, Any
import logging

# kg-gen 관련 임포트
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import dspy

# GraphRAG 관련 임포트
import graphrag

# 시각화 관련
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from pyvis.network import Network

# 경로 설정
PROJECT_ROOT = "<PROJECT_ROOT>"
KG_GEN_ROOT = os.path.join(PROJECT_ROOT, "kg_gen")
SNU_KG_OUTPUT = os.path.join(KG_GEN_ROOT, "snu_kg_output")

# 데이터 파일 경로 설정 (사용 가능한 파일 자동 선택)
possible_data_paths = [
    os.path.join(PROJECT_ROOT, "rag_dataset/processed_texts/final_texts.json")
]

# 존재하는 첫 번째 파일 선택
DATA_PATH = None
for path in possible_data_paths:
    if os.path.exists(path):
        DATA_PATH = path
        break

if DATA_PATH is None:
    print("❌ 데이터 파일을 찾을 수 없습니다.")
    print("다음 경로들을 확인했습니다:")
    for path in possible_data_paths:
        print(f"  - {path}")
    raise FileNotFoundError("No valid data file found")

# 출력 디렉토리 생성
os.makedirs(SNU_KG_OUTPUT, exist_ok=True)
os.makedirs(os.path.join(SNU_KG_OUTPUT, "intermediate"), exist_ok=True)
os.makedirs(os.path.join(SNU_KG_OUTPUT, "final"), exist_ok=True)
os.makedirs(os.path.join(SNU_KG_OUTPUT, "logs"), exist_ok=True)

print("📁 프로젝트 설정:")
print(f"   프로젝트 루트: {PROJECT_ROOT}")
print(f"   SNU-KG 출력: {SNU_KG_OUTPUT}")
print(f"   데이터 경로: {DATA_PATH}")
print(f"   파일명: {os.path.basename(DATA_PATH)}")

# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(SNU_KG_OUTPUT, "logs", "snu_kg.log")),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("snu_kg")

## 2. API 키 설정

In [ ]:
# OpenAI API 키 설정
import getpass

if "OPENAI_API_KEY" not in os.environ:
    print("⚠️ OpenAI API 키가 환경 변수에 설정되지 않았습니다.")
    print("\nAPI 키를 입력하세요 (입력 내용은 화면에 표시되지 않습니다):")
    api_key = getpass.getpass("OpenAI API Key: ")
    
    if api_key:
        os.environ['OPENAI_API_KEY'] = api_key
        print("✅ API 키가 설정되었습니다.")
    else:
        print("❌ API 키를 입력하지 않았습니다.")
        print("\n다음 중 하나의 방법으로 설정할 수 있습니다:")
        print("1. 이 셀을 다시 실행하여 API 키 입력")
        print("2. 터미널에서: export OPENAI_API_KEY='your-api-key'")
else:
    print("✅ API 키가 이미 설정되어 있습니다.")
    print("\n다시 설정하려면:")
    response = input("새 API 키를 입력하시겠습니까? (y/n): ")
    if response.lower() == 'y':
        api_key = getpass.getpass("새 OpenAI API Key: ")
        if api_key:
            os.environ['OPENAI_API_KEY'] = api_key
            print("✅ API 키가 업데이트되었습니다.")

# API 키 설정 확인 (처음 몇 자리만 표시)
if "OPENAI_API_KEY" in os.environ and os.environ['OPENAI_API_KEY']:
    key_preview = os.environ['OPENAI_API_KEY'][:8] + "..." if len(os.environ['OPENAI_API_KEY']) > 8 else "설정됨"
    print(f"\n현재 API 키: {key_preview}")

## 3. 통합 설정 파일 생성

In [ ]:
# 통합 설정 생성
snu_kg_config = {
    'pipeline': {
        'name': 'SNU-KG Unified Pipeline',
        'version': '1.0',
        'created_at': datetime.now().isoformat()
    },
    'graphrag': {
        'chunk_size': 300,
        'chunk_overlap': 100,
        'chunk_strategy': 'tokens',
        'model': 'gpt-4o-mini',
        'temperature': 0,
        'max_tokens': 16384,  # GPT-4의 최대 토큰 수로 수정 (50000 → 16384)
        'embedding_model': 'text-embedding-ada-002'
    },
    'kggen': {
        'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
        'clustering_threshold': 0.85,
        'min_cluster_size': 2,
        'dedup_temperature': 0,
        'context': '토마토, 고추, 파프리카 등 작물의 생장과 식물 질병, 그리고 정부의 작물 생장을 위한 기술지원 및 문제해결 과정에 초점을 맞춘 한국 농업 문서를 다룬다.'
    },
    'integration': {
        'use_kggen_clustering': True,
        'clustering_before_leiden': True,
        'preserve_cluster_info': True,
        'merge_similar_communities': True
    },
    'output': {
        'save_intermediate': True,
        'visualization': True,
        'format': 'parquet'  # GraphRAG 호환
    }
}

# 설정 저장
config_path = os.path.join(SNU_KG_OUTPUT, "snu_kg_config.yaml")
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.dump(snu_kg_config, f, default_flow_style=False, allow_unicode=True)

print("✅ 통합 설정 파일 생성 완료")
print(f"📄 설정 파일: {config_path}")
print(f"   - max_tokens: {snu_kg_config['graphrag']['max_tokens']} (GPT-4o-mini 호환)")

## 4. 파이프라인 시간 순서 및 프로세스

### 전체 파이프라인 흐름도:

```
[1. 데이터 준비] 
    ↓
[2. GraphRAG 초기화]
    ↓
[3. 프롬프트 튜닝] ← 한국 농업 도메인 특화
    ↓
[4. GraphRAG 인덱싱] ← 엔티티 추출 (title, type, description, source 등)
    ↓
[5. kg-gen 클러스터링 준비 (임베딩 생성)]
    ↓
[6. kg-gen 클러스터링] ← K-means + 중복 제거
    ↓
[8. 관계 업데이트] ← 클러스터링된 엔티티에 맞춰 parquet 업데이트
    ↓
[9. GraphRAG Leiden 커뮤니티 탐지] ← 계층적 구조 생성
    ↓
[10. 최종 그래프 저장]
```

### GraphRAG vs SNU-KG 타임라인:

| 단계 | GraphRAG 기본 | SNU-KG 통합 | 추가된 기능 |
|------|---------------|-------------|------------|
| 1. 엔티티 추출 | title, type, description | 동일 | - |
| 2. 엔티티 병합 | (title, type) 기준 | 동일 | - |
| 3. 요약 | 엔티티/관계 설명 요약 | 동일 | - |
| 4. **언어 통합 준비** | - | 클러스터링으로 언어 차이 해결 | ✨ 자동 언어 통합 |
| 5. **임베딩 생성** | - | 전체 메타데이터 임베딩 | ✨ 풍부한 문맥 |
| 6. **kg-gen 클러스터링** | - | K-means + LLM 중복제거 | ✨ 효율성 향상 |
| 7. 커뮤니티 탐지 | Leiden 알고리즘 | 클러스터링 후 Leiden | ✨ 정제된 데이터 |

### 핵심 차이점:
- **GraphRAG**: 엔티티 추출 → 병합 → 커뮤니티 탐지
- **SNU-KG**: 엔티티 추출 → 병합 → **이중언어 강화** → **kg-gen 클러스터링** → 커뮤니티 탐지

## 5. 데이터 준비

In [ ]:
# 데이터 로드 및 준비
print("📊 데이터 준비 중...")

# 입력 데이터 디렉토리 설정
graphrag_input_dir = os.path.join(SNU_KG_OUTPUT, "graphrag_input")
os.makedirs(graphrag_input_dir, exist_ok=True)

# 이전 입력 파일 정리
if os.path.exists(graphrag_input_dir):
    for file in os.listdir(graphrag_input_dir):
        os.remove(os.path.join(graphrag_input_dir, file))
    print("🧹 이전 입력 파일을 정리했습니다.")

# 데이터 로드
print(f"\n📁 데이터 파일 로드: {os.path.basename(DATA_PATH)}")
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

# 데이터 타입 확인
if isinstance(data, list):
    print(f"✅ {len(data)} 개의 문서 로드 완료")
    documents = data
elif isinstance(data, dict):
    # 딕셔너리인 경우 documents 키가 있는지 확인
    if 'documents' in data:
        documents = data['documents']
        print(f"✅ {len(documents)} 개의 문서 로드 완료 (documents 키에서)")
    else:
        # 단일 문서로 처리
        documents = [data]
        print(f"✅ 1개의 문서 로드 완료 (단일 문서)")
else:
    print(f"❌ 예상치 못한 데이터 형식: {type(data)}")
    raise ValueError(f"Unexpected data format: {type(data)}")

# GraphRAG 입력 형식으로 변환
doc_count = 0
for idx, item in enumerate(documents):
    # 텍스트 파일로 저장 (GraphRAG는 텍스트 파일을 입력으로 받음)
    file_path = os.path.join(graphrag_input_dir, f"doc_{idx:04d}.txt")
    
    # JSON 구조에 맞춰 텍스트 추출
    content = ""
    
    if isinstance(item, dict):
        # 'text' 필드가 있는 경우
        if 'text' in item:
            content = item['text']
        # 'title'과 'content' 필드가 분리된 경우
        elif 'title' in item or 'content' in item:
            if 'title' in item and item['title']:
                content = f"제목: {item['title']}\n\n"
            if 'content' in item and item['content']:
                content += f"내용: {item['content']}\n"
        else:
            # 기타 필드들을 모두 텍스트로 변환
            content = json.dumps(item, ensure_ascii=False, indent=2)
    elif isinstance(item, str):
        content = item
    else:
        print(f"⚠️ 문서 {idx}: 예상치 못한 형식 {type(item)}")
        continue
    
    # 빈 콘텐츠는 건너뛰기
    if not content or not content.strip():
        print(f"⚠️ 문서 {idx}: 빈 콘텐츠 건너뛰기")
        continue
    
    with open(file_path, 'w', encoding='utf-8') as f:
        f.write(content)
    doc_count += 1

print(f"\n✅ GraphRAG 입력 파일 생성 완료: {doc_count}개 파일")
print(f"📁 저장 위치: {graphrag_input_dir}")

# 생성된 파일 확인
if doc_count > 0:
    sample_files = sorted(os.listdir(graphrag_input_dir))[:min(3, doc_count)]
    print(f"\n📄 생성된 파일 샘플:")
    for file_name in sample_files:
        file_path = os.path.join(graphrag_input_dir, file_name)
        file_size = os.path.getsize(file_path)
        with open(file_path, 'r', encoding='utf-8') as f:
            content_preview = f.read()[:200].replace('\n', ' ')
            print(f"  - {file_name} ({file_size:,} bytes): {content_preview}...")
else:
    print("\n❌ 생성된 파일이 없습니다. 데이터 형식을 확인하세요.")

## 6. GraphRAG 워크스페이스 초기화

In [ ]:
# GraphRAG 워크스페이스 초기화
print("🚀 GraphRAG 워크스페이스 초기화...")

# GraphRAG 워크스페이스 설정
GRAPHRAG_WORKSPACE = os.path.join(SNU_KG_OUTPUT, "graphrag_workspace")
os.makedirs(GRAPHRAG_WORKSPACE, exist_ok=True)
os.chdir(GRAPHRAG_WORKSPACE)

# 기존 설정 파일 확인
settings_file = os.path.join(GRAPHRAG_WORKSPACE, "settings.yaml")

if not os.path.exists(settings_file):
    print("\n📦 새 GraphRAG 프로젝트 초기화 중...")
    
    # graphrag init 실행
    init_cmd = [sys.executable, "-m", "graphrag", "init", "--root", "."]
    
    try:
        result = subprocess.run(
            init_cmd,
            cwd=GRAPHRAG_WORKSPACE,
            capture_output=True,
            text=True,
            timeout=30
        )
        
        if result.returncode == 0:
            print("✅ 초기화 성공!")
        else:
            print(f"⚠️ 초기화 중 경고: {result.stderr}")
            
    except subprocess.TimeoutExpired:
        print("⏱️ 초기화 타임아웃 (30초)")
    except Exception as e:
        print(f"❌ 초기화 실패: {e}")
else:
    print("✅ 기존 워크스페이스 사용")

# 입력 파일 복사
workspace_input = os.path.join(GRAPHRAG_WORKSPACE, "input")
if os.path.exists(workspace_input):
    shutil.rmtree(workspace_input)
shutil.copytree(graphrag_input_dir, workspace_input)

print(f"✅ 입력 파일 복사 완료: {workspace_input}")

## 7. GraphRAG 설정 구성

In [ ]:
# GraphRAG 설정 수정
print("⚙️ GraphRAG 설정 구성...")

if os.path.exists(settings_file):
    with open(settings_file, 'r', encoding='utf-8') as f:
        settings = yaml.safe_load(f)
    
    print("📝 현재 설정을 수정합니다...")
    
    # 모델 설정
    if 'models' not in settings:
        settings['models'] = {}
    
    # 기본 채팅 모델 설정
    settings['models']['default_chat_model'] = {
        'api_key': '${OPENAI_API_KEY}',
        'type': 'openai_chat',
        'model_type': 'chat',  # GraphRAG v3를 위한 명시적 타입
        'model': snu_kg_config['graphrag']['model'],
        'model_supports_json': True,
        'max_tokens': snu_kg_config['graphrag']['max_tokens'],
        'temperature': snu_kg_config['graphrag']['temperature']
    }
    
    # 기본 임베딩 모델 설정
    settings['models']['default_embedding_model'] = {
        'api_key': '${OPENAI_API_KEY}',
        'type': 'openai_embedding',
        'model_type': 'embedding',  # GraphRAG v3를 위한 명시적 타입
        'model': snu_kg_config['graphrag']['embedding_model']
    }
    
    # 청킹 설정 (한국어 특성 고려)
    if 'chunks' not in settings:
        settings['chunks'] = {}
    
    settings['chunks']['size'] = snu_kg_config['graphrag']['chunk_size']
    settings['chunks']['overlap'] = snu_kg_config['graphrag']['chunk_overlap']
    settings['chunks']['strategy'] = snu_kg_config['graphrag']['chunk_strategy']
    
    # 그래프 추출 설정
    if 'extract_graph' not in settings:
        settings['extract_graph'] = {}
    
    settings['extract_graph']['model_id'] = 'default_chat_model'
    settings['extract_graph']['max_gleanings'] = 1  # 비용 절감

    # 공변량(claims) 추출 활성화
    if 'extract_claims' not in settings:
        settings['extract_claims'] = {}
    
    settings['extract_claims']['enabled'] = True
    settings['extract_claims']['model_id'] = 'default_chat_model'
    settings['extract_claims']['description'] = "한국 농업 관련 주요 주장, 사실, 통계 등을 추출합니다."
    print("✅ extract_claims (공변량 추출) 활성화")
    
    # 설정 저장
    with open(settings_file, 'w', encoding='utf-8') as f:
        yaml.dump(settings, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
    
    print("✅ settings.yaml 업데이트 완료")

# .env 파일 생성
env_file = os.path.join(GRAPHRAG_WORKSPACE, ".env")
env_content = f"""OPENAI_API_KEY={os.environ.get('OPENAI_API_KEY', '')}
GRAPHRAG_API_KEY={os.environ.get('OPENAI_API_KEY', '')}
"""

with open(env_file, 'w') as f:
    f.write(env_content)

print("✅ 환경 설정 완료")

## 8. 프롬프트 튜닝 (선택적)

GraphRAG의 프롬프트 튜닝은 기본 설정으로도 충분히 작동합니다.
프롬프트 튜닝을 건너뛰려면 다음 셀에서 `SKIP_PROMPT_TUNING = True`로 설정하세요.

In [ ]:
# 프롬프트 튜닝 건너뛰기 옵션
SKIP_PROMPT_TUNING = False  # True로 설정하면 프롬프트 튜닝을 건너뜁니다

if SKIP_PROMPT_TUNING:
    print("⏭️ 프롬프트 튜닝을 건너뜁니다.")
    print("   GraphRAG가 기본 프롬프트를 사용하여 정상 작동합니다.")
    print("\n" + "="*60)
else:
    # 프롬프트 튜닝
    print("🎯 한국 농업 도메인 특화 프롬프트 튜닝...")
    print("\n" + "="*60)
    
    # API 키 확인
    if "OPENAI_API_KEY" not in os.environ or not os.environ['OPENAI_API_KEY']:
        print("❌ OpenAI API 키가 설정되지 않았습니다.")
        print("   셀 5를 실행하여 API 키를 설정한 후 다시 시도하세요.")
        raise ValueError("OpenAI API key not set")
    
    # 프롬프트 디렉토리 설정
    prompts_dir = os.path.join(GRAPHRAG_WORKSPACE, "prompts")
    
    # 프롬프트 튜닝 명령어 구성
    print("\n📝 프롬프트 튜닝 설정:")
    print("   - 도메인: 한국 농업 (토마토, 고추, 파프리카 등)")
    print("   - 언어: 한국어")
    # print("   - 샘플 수: 20개 (빠른 처리)")
    # print("   - 최대 토큰: 1000 (비용 절감)")
    # print("   - 청크 크기: 300")
    
    prompt_tune_cmd = [
        sys.executable, "-m", "graphrag", "prompt-tune",
        "--root", ".",
        "--domain", "Covers Korean agricultural documents focusing on the growth and plant diseases of crops such as tomatoes, peppers, and paprika, as well as the government's technical support and problem-solving process for crop growth.",
        "--language", "Korean",
        "--selection-method", "auto",
        # "--limit", "20",
        "--max-tokens", "2000",
        "--chunk-size", str(snu_kg_config['graphrag']['chunk_size']),
        "--min-examples-required", "2",
        #"--n-subset-max", "200",
        #"--k", "20"
    ]
    
    print("\n💻 실행 명령어:")
    print(f"   {' '.join(prompt_tune_cmd)}")
    print("\n🏃 실행 중...")
    print("⏰ 예상 소요 시간: 10-20분")
    
    try:
        import threading
        import queue
        
        # 출력 큐
        output_queue = queue.Queue()
        stderr_queue = queue.Queue()
        
        # Non-blocking 출력을 위한 함수
        def enqueue_output(pipe, queue):
            for line in iter(pipe.readline, ''):
                queue.put(line)
            pipe.close()
        
        # 프로세스 시작
        process = subprocess.Popen(
            prompt_tune_cmd,
            cwd=GRAPHRAG_WORKSPACE,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=1,
            universal_newlines=True,
            env={**os.environ, 'PYTHONUNBUFFERED': '1'}
        )
        
        # 출력 스레드 시작
        stdout_thread = threading.Thread(target=enqueue_output, args=(process.stdout, output_queue))
        stdout_thread.daemon = True
        stdout_thread.start()
        
        stderr_thread = threading.Thread(target=enqueue_output, args=(process.stderr, stderr_queue))
        stderr_thread.daemon = True
        stderr_thread.start()
        
        # 진행 상황 모니터링
        start_time = time.time()
        last_output_time = start_time
        timeout_seconds = 1800  # 30분 타임아웃
        
        while True:
            # 프로세스가 종료되었는지 확인
            if process.poll() is not None:
                break
            
            # stdout 출력 확인
            try:
                line = output_queue.get_nowait()
                if line.strip():
                    # 주요 진행 상황만 표시
                    if any(keyword in line.lower() for keyword in ['loading', 'generating', 'creating', 'writing', 'done', 'complete', 'saved']):
                        print(f"   → {line.strip()}")
                    last_output_time = time.time()
            except queue.Empty:
                pass
            
            # stderr 확인 (디버깅용)
            try:
                line = stderr_queue.get_nowait()
                if line.strip() and "error" in line.lower():
                    print(f"   ⚠️ {line.strip()}")
            except queue.Empty:
                pass
            
            # 타임아웃 체크
            current_time = time.time()
            if current_time - last_output_time > timeout_seconds:
                print(f"\n⏱️ 타임아웃: {timeout_seconds}초 동안 응답이 없습니다.")
                process.terminate()
                break
            
            # 진행 시간 표시
            elapsed = current_time - start_time
            print(f"   ⏳ 진행 중... ({int(elapsed)}초)", end='\r')
            
            time.sleep(0.1)
        
        # 프로세스 종료 대기
        return_code = process.wait()
        elapsed_time = time.time() - start_time
        
        # 남은 출력 처리
        while not output_queue.empty():
            line = output_queue.get()
            if line.strip():
                print(f"   → {line.strip()}")
        
        if return_code == 0:
            print(f"\n✅ 프롬프트 튜닝 완료! (소요 시간: {int(elapsed_time)}초)")
            
        else:
            print(f"\n⚠️ 프롬프트 튜닝이 종료됨 (코드: {return_code})")
            
            # stderr 메시지 확인
            stderr_messages = []
            while not stderr_queue.empty():
                stderr_messages.append(stderr_queue.get().strip())
            
            if stderr_messages:
                print("\n📋 메시지:")
                for msg in stderr_messages[-5:]:  # 마지막 5개만
                    if msg:
                        print(f"   ! {msg}")
            
            print("\n💡 GraphRAG가 기본 프롬프트를 사용합니다.")
            
    except KeyboardInterrupt:
        print("\n\n⛔ 사용자가 중단했습니다.")
        if process.poll() is None:
            process.terminate()
        print("   기본 프롬프트를 사용하여 계속 진행할 수 있습니다.")
        
    except Exception as e:
        print(f"\n❌ 오류 발생: {e}")
        print("   기본 프롬프트를 사용합니다.")
    
    print("\n" + "="*60)

## 9. GraphRAG 인덱싱 실행

In [ ]:
# GraphRAG 인덱싱 실행 (엔티티/관계 추출만)
print("🏗️ GraphRAG 엔티티/관계 추출 시작...")
print(f"📁 작업 디렉토리: {GRAPHRAG_WORKSPACE}")
print("\n" + "="*60)

# settings.yaml 백업
settings_file = os.path.join(GRAPHRAG_WORKSPACE, "settings.yaml")
settings_backup = settings_file + ".backup"

if os.path.exists(settings_file):
    # 원본 설정 백업
    if not os.path.exists(settings_backup):
        shutil.copy(settings_file, settings_backup)
        print("📦 원본 settings.yaml 백업 완료")
    
    # 설정 파일 읽기
    with open(settings_file, 'r', encoding='utf-8') as f:
        settings = yaml.safe_load(f)
    
    # 엔티티/관계 추출 워크플로우만 설정
    settings['workflows'] = [
        "load_input_documents",
        "create_base_text_units",
        "create_final_documents",
        "extract_graph",
        "finalize_graph"
    ]
    
    # 수정된 설정 저장
    with open(settings_file, 'w', encoding='utf-8') as f:
        yaml.dump(settings, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
    
    print("✅ 엔티티/관계 추출만 실행하도록 설정 수정됨")
    print("   실행될 워크플로우:")
    for wf in settings['workflows']:
        print(f"     - {wf}")

# 인덱싱 명령어 (기본 명령어 사용)
index_cmd = [
    sys.executable, "-m", "graphrag", "index",
    "--root", ".",
    "--verbose"
]

print("\n💻 실행 명령어:")
print(f"   {' '.join(index_cmd)}")
print("\n🏃 엔티티/관계 추출 진행 중...")
print("⏰ 예상 소요 시간: 60-180분 (문서 수에 따라 변동)")
print("\n💡 커뮤니티 탐지는 kg-gen 중복 제거 후 별도로 실행됩니다.")

# 출력 디렉토리
output_dir = os.path.join(GRAPHRAG_WORKSPACE, "output")

start_time = time.time()

try:
    # 프로세스 실행
    process = subprocess.Popen(
        index_cmd,
        cwd=GRAPHRAG_WORKSPACE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )
    
    # 주요 진행 사항 표시
    for line in process.stdout:
        if line.strip():
            # 워크플로우 진행 상황 표시
            if any(keyword in line for keyword in ['workflow', 'executing', 'completed', 'error']):
                print(f"   → {line.strip()}")
    
    # 프로세스 종료 대기
    process.wait()
    
    if process.returncode == 0:
        print("\n🎉 엔티티/관계 추출 성공적으로 완료!")
    else:
        print("\n⚠️ 추출이 오류와 함께 종료됨")
        stderr_output = process.stderr.read()
        if stderr_output:
            print(f"오류 메시지: {stderr_output[:1000]}...")
            
except Exception as e:
    print(f"\n❌ 추출 중 오류 발생: {e}")

# 실행 시간 표시
elapsed_time = time.time() - start_time
print(f"\n⏱️ 총 소요 시간: {elapsed_time//60:.0f}분 {elapsed_time%60:.0f}초")
print("\n" + "="*60)

In [ ]:
# GraphRAG 결과 로드 - 커뮤니티 데이터 포함
print("📊 GraphRAG 결과 로드 중...")

entities_path = os.path.join(output_dir, "entities.parquet")
relations_path = os.path.join(output_dir, "relationships.parquet")
communities_path = os.path.join(output_dir, "communities.parquet")
community_reports_path = os.path.join(output_dir, "community_reports.parquet")

if os.path.exists(entities_path) and os.path.exists(relations_path):
    entities_df = pd.read_parquet(entities_path)
    relations_df = pd.read_parquet(relations_path)
    
    print(f"✅ 엔티티: {len(entities_df)}개")
    print(f"✅ 관계: {len(relations_df)}개")
    
    # 커뮤니티 데이터 로드
    if os.path.exists(communities_path):
        communities_df = pd.read_parquet(communities_path)
        print(f"✅ 커뮤니티: {len(communities_df)}개")
    else:
        print("⚠️ 커뮤니티 파일을 찾을 수 없습니다.")
        communities_df = None
    
    # 커뮤니티 리포트 로드
    if os.path.exists(community_reports_path):
        community_reports_df = pd.read_parquet(community_reports_path)
        print(f"✅ 커뮤니티 리포트: {len(community_reports_df)}개")
    else:
        print("⚠️ 커뮤니티 리포트 파일을 찾을 수 없습니다.")
        community_reports_df = None
else:
    print("❌ GraphRAG 출력 파일을 찾을 수 없습니다.")
    print("   인덱싱이 완료되었는지 확인하세요.")
    raise FileNotFoundError("GraphRAG output files not found")

# 엔티티 데이터 확인
print("\n📋 엔티티 컬럼:", entities_df.columns.tolist())
print("\n🔍 샘플 엔티티 (처음 3개):")
for idx, row in entities_df.head(3).iterrows():
    print(f"\n[{idx}]")
    print(f"   Title: {row.get('title', 'N/A')}")
    print(f"   Type: {row.get('type', 'N/A')}")
    print(f"   Description: {row.get('description', 'N/A')[:100]}...")

# 관계 데이터 확인
print("\n📋 관계 컬럼:", relations_df.columns.tolist())
print("\n🔍 샘플 관계 (처음 3개):")
for idx, row in relations_df.head(3).iterrows():
    print(f"\n[{idx}]")
    print(f"   Source: {row.get('source', 'N/A')}")
    print(f"   Target: {row.get('target', 'N/A')}")
    print(f"   Description: {row.get('description', 'N/A')[:100]}...")

# 커뮤니티 데이터 확인
if communities_df is not None:
    print("\n📋 커뮤니티 컬럼:", communities_df.columns.tolist())
    print("\n🔍 샘플 커뮤니티 (처음 3개):")
    for idx, row in communities_df.head(3).iterrows():
        print(f"\n[{idx}]")
        print(f"   Community ID: {row.get('community', 'N/A')}")
        print(f"   Level: {row.get('level', 'N/A')}")
        print(f"   Title: {row.get('title', 'N/A')}")
        print(f"   Size: {row.get('size', 'N/A')}")
        print(f"   Entity IDs: {str(row.get('entity_ids', 'N/A'))[:100]}...")
        
# 커뮤니티 리포트 확인
if community_reports_df is not None:
    print("\n📋 커뮤니티 리포트 컬럼:", community_reports_df.columns.tolist())
    print("\n🔍 샘플 커뮤니티 리포트 (처음 2개):")
    for idx, row in community_reports_df.head(2).iterrows():
        print(f"\n[{idx}]")
        print(f"   Community ID: {row.get('community', 'N/A')}")
        print(f"   Level: {row.get('level', 'N/A')}")
        print(f"   Title: {row.get('title', 'N/A')}")
        print(f"   Summary: {row.get('summary', 'N/A')[:200]}...")
        
# 커뮤니티 통계 분석
if communities_df is not None:
    print("\n📊 커뮤니티 통계 분석:")
    
    # 레벨별 커뮤니티 수
    level_counts = communities_df['level'].value_counts().sort_index()
    print(f"\n   레벨별 커뮤니티 수:")
    for level, count in level_counts.items():
        print(f"     Level {level}: {count}개")
    
    # 커뮤니티 크기 분포
    if 'size' in communities_df.columns:
        size_stats = communities_df.groupby('level')['size'].describe()
        print(f"\n   커뮤니티 크기 통계:")
        print(size_stats)
    
    # 가장 큰 커뮤니티
    if 'size' in communities_df.columns:
        largest_communities = communities_df.nlargest(5, 'size')
        print(f"\n   가장 큰 5개 커뮤니티:")
        for idx, row in largest_communities.iterrows():
            print(f"     Community {row['community']} (Level {row['level']}): {row['size']}개 엔티티")
            if 'title' in row and pd.notna(row['title']):
                print(f"       Title: {row['title']}")

## 10. kg-gen 메커니즘을 활용한 엔티티/관계 통합

이 섹션에서는 kg-gen의 핵심 메커니즘(임베딩 + K-means + DSPy)을 차용하여 GraphRAG의 엔티티와 관계를 통합합니다.
GraphRAG의 풍부한 메타데이터 구조를 완전히 보존하면서 중복을 제거합니다.

### 10-1. 데이터 로드 및 구조 분석 (A-1)

In [ ]:
# A-1. 데이터 로드 및 구조 분석
print("="*60)
print("A-1. 데이터 로드 및 구조 분석 시작")
print("="*60)

# GraphRAG 출력 경로 설정
output_dir = os.path.join(GRAPHRAG_WORKSPACE, "output")
entities_path = os.path.join(output_dir, "entities.parquet")
relationships_path = os.path.join(output_dir, "relationships.parquet")

# 엔티티 데이터 로드
entities_df = pd.read_parquet(entities_path)
print(f"\n✅ 원본 엔티티 수: {len(entities_df)}")
print(f"✅ 엔티티 컬럼: {entities_df.columns.tolist()}")

# 관계 데이터 로드
relationships_df = pd.read_parquet(relationships_path)
print(f"\n✅ 원본 관계 수: {len(relationships_df)}")
print(f"✅ 관계 컬럼: {relationships_df.columns.tolist()}")

# text_unit_ids 형식 확인
print("\n📋 text_unit_ids 형식 확인:")
sample_entity = entities_df.iloc[0]
print(f"  - 첫 번째 엔티티의 text_unit_ids: {sample_entity['text_unit_ids']}")
print(f"  - 타입: {type(sample_entity['text_unit_ids'])}")

# text_unit_ids가 리스트인지 확인
if isinstance(sample_entity['text_unit_ids'], list):
    print("  ✅ text_unit_ids는 이미 리스트 형태입니다.")
else:
    print("  ⚠️ text_unit_ids가 리스트가 아닙니다. 변환이 필요할 수 있습니다.")

# 한국어-영어 중복 가능성 있는 엔티티 예시 찾기
print("\n🔍 한국어-영어 중복 가능성 있는 엔티티 예시:")
korean_entities = entities_df[entities_df['title'].str.contains('[가-힣]', regex=True)]
english_entities = entities_df[~entities_df['title'].str.contains('[가-힣]', regex=True)]

print(f"  - 한국어 엔티티: {len(korean_entities)}개")
print(f"  - 영어/기타 엔티티: {len(english_entities)}개")

# 샘플 출력
print("\n📄 한국어 엔티티 샘플 (처음 5개):")
for idx, row in korean_entities.head(5).iterrows():
    print(f"  [{row['human_readable_id']}] {row['title']} ({row['type']})")

print("\n📄 영어 엔티티 샘플 (처음 5개):")
for idx, row in english_entities.head(5).iterrows():
    print(f"  [{row['human_readable_id']}] {row['title']} ({row['type']})")

# 통계 요약
print("\n📊 데이터 요약:")
print(f"  - 총 엔티티 수: {len(entities_df)}")
print(f"  - 총 관계 수: {len(relationships_df)}")
print(f"  - 유니크한 엔티티 타입: {entities_df['type'].nunique()}개")
print(f"  - 엔티티 타입 분포:")
for entity_type, count in entities_df['type'].value_counts().head(10).items():
    print(f"    • {entity_type}: {count}개")

print("\n✅ A-1 단계 완료: 데이터 구조 분석 완료")

### 10-2. kg-gen 스타일 임베딩 생성 (A-2)

In [ ]:
# A-2. kg-gen 스타일 임베딩 생성
print("="*60)
print("A-2. kg-gen 스타일 임베딩 생성 시작")
print("="*60)

# text_unit_ids 처리를 위한 함수 정의
def merge_text_unit_ids(arrays):
    """여러 numpy array 또는 리스트의 text_unit_ids를 합집합으로 병합"""
    all_ids = set()
    for arr in arrays:
        if isinstance(arr, np.ndarray):
            all_ids.update(arr.tolist())
        elif isinstance(arr, list):
            all_ids.update(arr)
    return list(all_ids)

# kg-gen의 임베딩 방식 차용
from sentence_transformers import SentenceTransformer

print("\n📚 임베딩 모델 로드 중...")
model = SentenceTransformer(snu_kg_config['kggen']['embedding_model'])
print(f"✅ 모델 로드 완료: {snu_kg_config['kggen']['embedding_model']}")

# 엔티티 텍스트 생성 (kg-gen처럼 간단하게 title | type만 사용)
print("\n🔤 엔티티 텍스트 생성 중...")
entity_texts = entities_df.apply(
    lambda row: f"{row['title']} | {row['type']}", 
    axis=1
).tolist()

# 몇 가지 샘플 출력
print("\n📝 엔티티 텍스트 샘플:")
for i in range(min(5, len(entity_texts))):
    print(f"  [{i}] {entity_texts[i]}")

# 임베딩 생성
print(f"\n🧮 {len(entity_texts)}개 엔티티에 대한 임베딩 생성 중...")
embeddings = model.encode(entity_texts, show_progress_bar=True)

print(f"\n✅ 임베딩 생성 완료!")
print(f"  - 임베딩 shape: {embeddings.shape}")
print(f"  - 임베딩 차원: {embeddings.shape[1]}")

# 임베딩 저장 (나중에 사용하기 위해)
embeddings_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "entity_embeddings.npy")
np.save(embeddings_path, embeddings)
print(f"\n💾 임베딩 저장 완료: {embeddings_path}")

print("\n✅ A-2 단계 완료: kg-gen 스타일 임베딩 생성 완료")

### 10-3. kg-gen 스타일 K-means 클러스터링 (A-3)

In [ ]:
# A-3. kg-gen 스타일 K-means 클러스터링
print("="*60)
print("A-3. kg-gen 스타일 K-means 클러스터링 시작")
print("="*60)

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# kg-gen의 클러스터링 방식 차용
print("\n🎯 클러스터링 설정:")
cluster_size = 30  # 임시값 / 128이 kg-gen default
n_clusters = max(1, len(entities_df) // cluster_size)
print(f"  - 총 엔티티 수: {len(entities_df)}")
print(f"  - 클러스터 크기 기준: {cluster_size}")
print(f"  - 클러스터 수: {n_clusters}")

# 클러스터 수가 1이면 모든 엔티티가 하나의 클러스터에
if n_clusters == 1:
    print("\n⚠️ 엔티티 수가 적어 모든 엔티티를 하나의 클러스터로 처리합니다.")
    clusters = np.zeros(len(entities_df), dtype=int)
else:
    # K-means 실행
    print(f"\n🔄 K-means 클러스터링 실행 중 (k={n_clusters})...")
    kmeans = KMeans(
        n_clusters=n_clusters, 
        random_state=42,
        n_init=10,
        max_iter=300
    )
    clusters = kmeans.fit_predict(embeddings)
    
    # Silhouette score 계산 (클러스터 품질 평가)
    if n_clusters > 1:
        silhouette_avg = silhouette_score(embeddings, clusters)
        print(f"\n📊 클러스터링 품질:")
        print(f"  - Silhouette Score: {silhouette_avg:.3f}")
        print(f"    (1에 가까울수록 좋음, -1에 가까울수록 나쁨)")

# 클러스터별 엔티티 그룹화
cluster_groups = {}
for idx, cluster_id in enumerate(clusters):
    if cluster_id not in cluster_groups:
        cluster_groups[cluster_id] = []
    cluster_groups[cluster_id].append(idx)

# 클러스터 통계
print(f"\n📈 클러스터링 결과:")
print(f"  - 생성된 클러스터 수: {len(cluster_groups)}")
print(f"  - 클러스터별 크기:")

cluster_sizes = []
for cluster_id, entity_indices in sorted(cluster_groups.items()):
    size = len(entity_indices)
    cluster_sizes.append(size)
    if cluster_id < 5:  # 처음 5개 클러스터만 표시
        print(f"    • 클러스터 {cluster_id}: {size}개 엔티티")

if len(cluster_groups) > 5:
    print(f"    • ... (총 {len(cluster_groups)}개 클러스터)")

# 클러스터 크기 통계
print(f"\n  - 클러스터 크기 통계:")
print(f"    • 최소: {min(cluster_sizes)}개")
print(f"    • 최대: {max(cluster_sizes)}개")
print(f"    • 평균: {np.mean(cluster_sizes):.1f}개")
print(f"    • 중앙값: {np.median(cluster_sizes):.0f}개")

# 각 클러스터 내의 엔티티 예시 출력
print("\n🔍 클러스터별 엔티티 예시:")
for cluster_id in range(min(3, len(cluster_groups))):  # 처음 3개 클러스터만
    entity_indices = cluster_groups[cluster_id]
    print(f"\n  클러스터 {cluster_id} ({len(entity_indices)}개 엔티티):")
    
    # 클러스터 내 처음 3개 엔티티 표시
    for i in range(min(3, len(entity_indices))):
        idx = entity_indices[i]
        entity = entities_df.iloc[idx]
        print(f"    - [{entity['human_readable_id']}] {entity['title']} ({entity['type']})")
    
    if len(entity_indices) > 3:
        print(f"    - ... 외 {len(entity_indices) - 3}개")

# 클러스터 정보 저장
cluster_info = {
    'clusters': clusters,
    'cluster_groups': cluster_groups,
    'n_clusters': n_clusters,
    'cluster_sizes': cluster_sizes
}

# 중간 결과 저장
import pickle
cluster_info_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "cluster_info.pkl")
with open(cluster_info_path, 'wb') as f:
    pickle.dump(cluster_info, f)
print(f"\n💾 클러스터 정보 저장 완료: {cluster_info_path}")

print("\n✅ A-3 단계 완료: K-means 클러스터링 완료")

### 10-4. DSPy 프롬프트 기반 중복 제거 (A-4)

In [ ]:
# A-4. DSPy 프롬프트 기반 중복 제거
print("="*60)
print("A-4. DSPy 프롬프트 기반 중복 제거 시작")
print("="*60)

import dspy

# DSPy 설정
print("\n🔧 DSPy 설정 중...")
# DSPy v3에서는 LM 클래스 사용
lm = dspy.LM(
    model=f"openai/{snu_kg_config['graphrag']['model']}",
    temperature=snu_kg_config['kggen']['dedup_temperature'],
    max_tokens=1000
)
dspy.configure(lm=lm)
print(f"✅ DSPy 설정 완료: {snu_kg_config['graphrag']['model']}")

# 프롬프트 파일 로드
prompt_file = "<PROJECT_ROOT>/kg_gen/kg-gen_prompt.txt"
if os.path.exists(prompt_file):
    print(f"\n📄 프롬프트 파일 로드: {prompt_file}")
    with open(prompt_file, 'r', encoding='utf-8') as f:
        prompt_content = f.read()
    # 엔티티 병합 프롬프트 추출 (수정된 라인 번호)
    entity_merge_prompt = prompt_content.split('\n')[2:123]  # 0-based index
    entity_merge_prompt_text = '\n'.join(entity_merge_prompt).strip()
    print("\n📋 엔티티 병합 프롬프트 로드 완료")
    print(f"   프롬프트 길이: {len(entity_merge_prompt_text)} 문자")
else:
    print(f"\n⚠️ 프롬프트 파일을 찾을 수 없습니다. 기본 프롬프트 사용")
    entity_merge_prompt_text = """한국 농업 문서의 엔티티 중복 제거를 위한 프롬프트입니다.
    동일한 개념을 나타내는 다양한 표현(한국어/영어, 약어, 동의어 등)을 찾아 병합합니다.
    예: 토마토/TOMATO, 병해충/pest disease, 온도/temperature 등"""

# DSPy Signature 정의 (수정: duplicates -> merge_candidates)
class EntityDeduplicate(dspy.Signature):
    __doc__ = entity_merge_prompt_text
    
    item: str = dspy.InputField(desc="병합 여부를 확인할 대상 엔티티")
    candidates: list[str] = dspy.InputField(desc="클러스터 내 다른 엔티티들")
    merge_candidates: list[str] = dspy.OutputField(desc="item과 병합해야 할 엔티티 리스트 (병합이 적절한 경우만)")
    reason: str = dspy.OutputField(desc="병합 결정 이유")

# 병합 그룹을 저장할 리스트
# 각 그룹: (대표 엔티티 인덱스, [병합될 엔티티 인덱스들])
merge_groups = []
processed_indices = set()

# 병합 이유와 상세 정보를 저장할 리스트
merge_reasons = []

print("\n🔍 클러스터별 중복 제거 시작...")
total_clusters = len(cluster_groups)

# 진행 상황 추적 변수
total_comparisons = 0
total_api_calls = 0

# 각 클러스터별로 중복 제거
for cluster_idx, (cluster_id, entity_indices) in enumerate(cluster_groups.items()):
    if len(entity_indices) <= 1:
        continue  # 1개 이하면 병합할 게 없음
    
    print(f"\n  클러스터 {cluster_id} 처리 중... ({cluster_idx + 1}/{total_clusters})")
    print(f"    - 엔티티 수: {len(entity_indices)}개")
    
    # 클러스터 내 엔티티 정보 준비
    cluster_entities = []
    for idx in entity_indices:
        entity = entities_df.iloc[idx]
        entity_info = f"{entity['title']} ({entity['type']})"
        cluster_entities.append((idx, entity_info))
    
    # 이미 처리된 엔티티는 제외
    unprocessed_entities = [
        (idx, info) for idx, info in cluster_entities 
        if idx not in processed_indices
    ]
    
    # 클러스터 내에서 중복 찾기
    cluster_merge_groups = []
    cluster_comparisons = 0
    
    for i, (current_idx, current_info) in enumerate(unprocessed_entities):
        if current_idx in processed_indices:
            continue
            
        # 나머지 엔티티들과 비교
        other_entities = [info for idx, info in unprocessed_entities[i+1:] 
                         if idx not in processed_indices]
        
        if not other_entities:
            continue
        
        cluster_comparisons += 1
        total_comparisons += 1
        
        try:
            # DSPy를 사용하여 병합 대상 찾기
            deduplicate = dspy.Predict(EntityDeduplicate)
            result = deduplicate(
                item=current_info,
                candidates=other_entities
            )
            total_api_calls += 1
            
            # 병합할 엔티티가 있으면 (merge_candidates로 변경)
            if result.merge_candidates and len(result.merge_candidates) > 0:
                # 병합 대상 엔티티의 인덱스 찾기
                merge_indices = []
                for merge_candidate in result.merge_candidates:
                    for idx, info in unprocessed_entities[i+1:]:
                        if info == merge_candidate and idx not in processed_indices:
                            merge_indices.append(idx)
                            processed_indices.add(idx)
                            break
                
                if merge_indices:
                    # 현재 엔티티도 처리됨으로 표시
                    processed_indices.add(current_idx)
                    
                    # human_readable_id가 가장 낮은 엔티티를 대표로
                    all_indices = [current_idx] + merge_indices
                    representative_idx = min(all_indices, 
                                           key=lambda x: entities_df.iloc[x]['human_readable_id'])
                    
                    merge_groups.append((representative_idx, all_indices))
                    cluster_merge_groups.append((representative_idx, all_indices))
                    
                    # 병합 상세 정보 수집 (NumPy 타입을 Python 네이티브 타입으로 변환)
                    merge_detail = {
                        'group_id': len(merge_groups) - 1,
                        'representative': {
                            'index': int(representative_idx),
                            'human_readable_id': int(entities_df.iloc[representative_idx]['human_readable_id']),
                            'title': str(entities_df.iloc[representative_idx]['title']),
                            'type': str(entities_df.iloc[representative_idx]['type']),
                            'description': str(entities_df.iloc[representative_idx]['description'])
                        },
                        'merged_entities': [],
                        'reason': str(result.reason),
                        'cluster_id': int(cluster_id),
                        'timestamp': pd.Timestamp.now().isoformat()
                    }
                    
                    # 병합될 엔티티들의 상세 정보
                    for idx in all_indices:
                        entity_detail = {
                            'index': int(idx),
                            'human_readable_id': int(entities_df.iloc[idx]['human_readable_id']),
                            'title': str(entities_df.iloc[idx]['title']),
                            'type': str(entities_df.iloc[idx]['type']),
                            'description': str(entities_df.iloc[idx]['description']),
                            'frequency': int(entities_df.iloc[idx]['frequency']),
                            'text_unit_ids': entities_df.iloc[idx]['text_unit_ids'].tolist() if hasattr(entities_df.iloc[idx]['text_unit_ids'], 'tolist') else list(entities_df.iloc[idx]['text_unit_ids'])
                        }
                        merge_detail['merged_entities'].append(entity_detail)
                    
                    merge_reasons.append(merge_detail)
                    
                    # 병합 정보 출력
                    print(f"\n    ✅ 병합 그룹 발견:")
                    print(f"       대표: [{entities_df.iloc[representative_idx]['human_readable_id']}] {entities_df.iloc[representative_idx]['title']}")
                    print(f"       병합 대상:")
                    for idx in all_indices:
                        if idx != representative_idx:
                            print(f"         - [{entities_df.iloc[idx]['human_readable_id']}] {entities_df.iloc[idx]['title']}")
                    print(f"       이유: {result.reason[:100]}...")
                    
        except Exception as e:
            print(f"    ⚠️ DSPy 처리 중 오류: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    if cluster_merge_groups:
        print(f"    → 클러스터 {cluster_id}에서 {len(cluster_merge_groups)}개 병합 그룹 생성")
    else:
        print(f"    → 클러스터 {cluster_id}에서 병합할 엔티티 없음")
    
    print(f"    → 클러스터 내 비교 수: {cluster_comparisons}")

# 병합 통계
print(f"\n📊 중복 제거 결과:")
print(f"  - 처리한 클러스터 수: {total_clusters}개")
print(f"  - 총 비교 수: {total_comparisons}")
print(f"  - API 호출 수: {total_api_calls}")
print(f"  - 총 병합 그룹 수: {len(merge_groups)}")
total_entities_to_merge = sum(len(indices) for _, indices in merge_groups)
print(f"  - 병합될 엔티티 수: {total_entities_to_merge}")
if total_entities_to_merge > 0:
    print(f"  - 병합 후 엔티티 수: {len(entities_df) - total_entities_to_merge + len(merge_groups)}")
    print(f"  - 감소율: {((total_entities_to_merge - len(merge_groups)) / len(entities_df) * 100):.1f}%")
else:
    print(f"  - 병합 후 엔티티 수: {len(entities_df)} (변화 없음)")

# 병합 정보 저장
merge_info = {
    'merge_groups': merge_groups,
    'processed_indices': processed_indices,
    'total_entities': len(entities_df),
    'entities_after_merge': len(entities_df) - total_entities_to_merge + len(merge_groups) if total_entities_to_merge > 0 else len(entities_df),
    'total_comparisons': total_comparisons,
    'total_api_calls': total_api_calls
}

merge_info_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "merge_info.pkl")
with open(merge_info_path, 'wb') as f:
    pickle.dump(merge_info, f)
print(f"\n💾 병합 정보 저장 완료: {merge_info_path}")

# 병합 이유와 상세 정보 저장
merge_reasons_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "merged_entities_reasons.json")
with open(merge_reasons_path, 'w', encoding='utf-8') as f:
    json.dump(merge_reasons, f, ensure_ascii=False, indent=2)
print(f"💾 병합 이유 및 상세 정보 저장 완료: {merge_reasons_path}")
print(f"   - 저장된 병합 그룹 수: {len(merge_reasons)}개")

# 병합 이유 요약 출력
if merge_reasons:
    print("\n📋 병합 이유 요약:")
    for i, reason in enumerate(merge_reasons[:3]):  # 처음 3개만 출력
        print(f"\n  [{i+1}] {reason['representative']['title']} 그룹")
        print(f"      병합 엔티티 수: {len(reason['merged_entities'])}개")
        print(f"      이유: {reason['reason'][:150]}...")
    if len(merge_reasons) > 3:
        print(f"\n  ... 외 {len(merge_reasons) - 3}개 그룹")

print("\n✅ A-4 단계 완료: DSPy 기반 중복 제거 완료")

### 10-5. GraphRAG 메타데이터 보존하며 병합 (A-5)

In [ ]:
# A-5. GraphRAG 메타데이터 보존하며 병합
print("="*60)
print("A-5. GraphRAG 메타데이터 보존하며 병합 시작")
print("="*60)

# 병합 정보 로드
with open(merge_info_path, 'rb') as f:
    merge_info = pickle.load(f)

merge_groups = merge_info['merge_groups']
print(f"\n📊 병합할 그룹 수: {len(merge_groups)}")

# 새로운 엔티티 데이터프레임 준비
merged_entities = []
processed_entity_indices = set()

# 병합 그룹별로 처리
for group_idx, (representative_idx, indices) in enumerate(merge_groups):
    print(f"\n  그룹 {group_idx + 1}/{len(merge_groups)} 처리 중...")
    
    # 병합될 엔티티들 가져오기
    entities_to_merge = [entities_df.iloc[idx] for idx in indices]
    
    # 대표 엔티티 (human_readable_id가 가장 낮은 것)
    representative = entities_df.iloc[representative_idx]
    
    # 병합된 엔티티 생성
    merged_entity = {
        'id': representative['id'],  # 대표 엔티티의 ID 사용
        'human_readable_id': representative['human_readable_id'],  # 가장 낮은 human_readable_id
        'title': representative['title'],  # 대표 엔티티의 title
        'type': representative['type'],  # 대표 엔티티의 type
        'description': '',  # 모든 description 병합 (아래에서 처리)
        'text_unit_ids': [],  # 모든 text_unit_ids 합집합 (아래에서 처리)
        'frequency': 0,  # 모든 frequency 합산 (아래에서 처리)
        'degree': representative['degree'],  # 일단 대표값 사용 (나중에 재계산)
        'x': representative['x'],  # 좌표는 대표값 사용
        'y': representative['y']
    }
    
    # Description 병합 (중요: 모든 원본 정보 보존)
    descriptions = []
    entity_num = f"{len(entities_to_merge)}개의 엔티티가 병합된 정보입니다. 설명에는 각 원본에 대한 정보가 포함됩니다."
    descriptions.append(entity_num)
    entity_to_merge_idx = 1
    for entity in entities_to_merge:
        desc = entity['description']
        if desc and str(desc).strip() and str(desc) != 'nan':
            # 원본 엔티티 정보를 포함하여 description 구성
            entity_info = f"[{entity_to_merge_idx}번 엔티티: title:{entity['title']}, type: {entity['type']} (text_unit_ids: {entity['text_unit_ids']}) {desc}]"
            descriptions.append(entity_info)
            entity_to_merge_idx += 1
    
    # 중복 제거하면서 모든 description 병합
    unique_descriptions = list(dict.fromkeys(descriptions))  # 순서 유지하며 중복 제거
    merged_entity['description'] = ' || '.join(unique_descriptions)
    
    # text_unit_ids 병합 (합집합)
    merged_entity['text_unit_ids'] = merge_text_unit_ids(
        [entity['text_unit_ids'] for entity in entities_to_merge]
    )
    
    # frequency 합산
    merged_entity['frequency'] = sum(entity['frequency'] for entity in entities_to_merge)
    
    # 병합 정보 출력
    print(f"    대표: [{representative['human_readable_id']}] {representative['title']}")
    print(f"    병합 엔티티 수: {len(entities_to_merge)}")
    print(f"    병합 후 frequency: {merged_entity['frequency']}")
    print(f"    text_unit_ids 수: {len(merged_entity['text_unit_ids'])}")
    
    merged_entities.append(merged_entity)
    processed_entity_indices.update(indices)

# 병합되지 않은 엔티티들 추가
print("\n📝 병합되지 않은 엔티티 처리 중...")
unmerged_count = 0
for idx, entity in entities_df.iterrows():
    if idx not in processed_entity_indices:
        merged_entities.append(entity.to_dict())
        unmerged_count += 1

print(f"  - 병합되지 않은 엔티티: {unmerged_count}개")

# 새 데이터프레임 생성
entities_merged_df = pd.DataFrame(merged_entities)

# 컬럼 순서 유지
entities_merged_df = entities_merged_df[entities_df.columns]

# 통계 출력
print(f"\n📊 병합 결과:")
print(f"  - 원본 엔티티 수: {len(entities_df)}")
print(f"  - 병합 후 엔티티 수: {len(entities_merged_df)}")
print(f"  - 감소율: {((len(entities_df) - len(entities_merged_df)) / len(entities_df) * 100):.1f}%")

# 병합 전후 비교 샘플
if len(merge_groups) > 0:
    print("\n🔍 병합 예시 (첫 번째 그룹):")
    first_group_rep_idx, first_group_indices = merge_groups[0]
    print(f"  병합 전:")
    for idx in first_group_indices[:3]:  # 최대 3개만
        entity = entities_df.iloc[idx]
        print(f"    - [{entity['human_readable_id']}] {entity['title']} ({entity['type']})")
    if len(first_group_indices) > 3:
        print(f"    - ... 외 {len(first_group_indices) - 3}개")
    
    # 병합 후 엔티티 찾기
    merged_entity = entities_merged_df[
        entities_merged_df['human_readable_id'] == entities_df.iloc[first_group_rep_idx]['human_readable_id']
    ].iloc[0]
    print(f"  병합 후:")
    print(f"    - [{merged_entity['human_readable_id']}] {merged_entity['title']} ({merged_entity['type']})")
    print(f"    - Frequency: {merged_entity['frequency']}")
    print(f"    - Description 길이: {len(merged_entity['description'])} 문자")

# 병합된 엔티티 저장
# human_readable_id 기준으로 정렬
entities_merged_df = entities_merged_df.sort_values("human_readable_id").reset_index(drop=True)

# 병합된 엔티티 저장
entities_merged_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "entities_merged.parquet")
entities_merged_df.to_parquet(entities_merged_path, index=False)
print(f"\n💾 병합된 엔티티 저장 완료: {entities_merged_path}")

print("\n✅ A-5 단계 완료: GraphRAG 메타데이터 보존 병합 완료")

### 10-6. 관계 업데이트 (A-6)

In [ ]:
# A-6. 관계 업데이트
print("="*60)
print("A-6. 관계 업데이트 시작")
print("="*60)

# 엔티티 title 매핑 생성 (병합 전 -> 병합 후)
entity_title_mapping = {}
for rep_idx, indices in merge_groups:
    rep_entity = entities_df.iloc[rep_idx]
    for idx in indices:
        old_entity = entities_df.iloc[idx]
        entity_title_mapping[old_entity['title']] = rep_entity['title']

print(f"\n📋 엔티티 매핑 생성 완료: {len(entity_title_mapping)}개 매핑")

# 관계 업데이트
print("\n🔄 관계 업데이트 중...")
updated_relationships = []
relationship_updates = 0

for idx, rel in relationships_df.iterrows():
    updated_rel = rel.to_dict()
    
    # Source 업데이트
    if rel['source'] in entity_title_mapping:
        updated_rel['source'] = entity_title_mapping[rel['source']]
        relationship_updates += 1
    
    # Target 업데이트
    if rel['target'] in entity_title_mapping:
        updated_rel['target'] = entity_title_mapping[rel['target']]
        relationship_updates += 1
    
    updated_relationships.append(updated_rel)

# 새 관계 데이터프레임 생성
relationships_updated_df = pd.DataFrame(updated_relationships)

# 중복 관계 처리 (동일한 source-target 쌍)
print("\n🔍 중복 관계 처리 중...")
original_rel_count = len(relationships_updated_df)

# source, target으로 그룹화
grouped = relationships_updated_df.groupby(['source', 'target'])

# 중복 관계 병합
merged_relationships = []
duplicate_count = 0
duplicate_examples = []

for (source, target), group in grouped:
    if len(group) > 1:
        # 중복 관계가 있는 경우
        duplicate_count += 1
        
        # 중복 관계 정보 수집 (처음 5개만 예시로 저장)
        if len(duplicate_examples) < 5:
            duplicate_info = {
                'source': source,
                'target': target,
                'count': len(group),
                'descriptions': group['description'].tolist()
            }
            duplicate_examples.append(duplicate_info)
        
        merged_rel = {
            'id': group.iloc[0]['id'],  # 첫 번째 ID 사용
            'human_readable_id': group['human_readable_id'].min(),  # 가장 낮은 ID
            'source': source,
            'target': target,
            'description': ' || '.join([str(d) for d in group['description'] if pd.notna(d)]),
            'weight': group['weight'].sum(),  # weight 합산
            'combined_degree': group['combined_degree'].max(),  # 최대값 사용
            'text_unit_ids': merge_text_unit_ids(group['text_unit_ids'].tolist())
        }
        merged_relationships.append(merged_rel)
    else:
        # 중복이 없는 경우
        merged_relationships.append(group.iloc[0].to_dict())

# 최종 관계 데이터프레임
relationships_merged_df = pd.DataFrame(merged_relationships)

# 컬럼 순서 유지
relationships_merged_df = relationships_merged_df[relationships_df.columns]

# 중복 관계 상세 정보 출력
if duplicate_count > 0:
    print(f"\n🔍 발견된 중복 관계: {duplicate_count}개")
    print("\n📋 중복 관계 예시 (최대 5개):")
    for i, dup in enumerate(duplicate_examples):
        print(f"\n  [{i+1}] {dup['source']} → {dup['target']}")
        print(f"      중복 수: {dup['count']}개")
        print(f"      설명들:")
        for j, desc in enumerate(dup['descriptions']):
            if pd.notna(desc):
                desc_preview = desc[:100] + "..." if len(str(desc)) > 100 else desc
                print(f"        {j+1}. {desc_preview}")

print(f"\n📊 관계 업데이트 결과:")
print(f"  - 원본 관계 수: {len(relationships_df)}")
print(f"  - 업데이트된 엔티티 참조: {relationship_updates}개")
print(f"  - 중복 제거 전 관계 수: {original_rel_count}")
print(f"  - 중복 제거 후 관계 수: {len(relationships_merged_df)}")
print(f"  - 제거된 중복 관계: {original_rel_count - len(relationships_merged_df)}개")

# 업데이트 예시
if relationship_updates > 0:
    print("\n🔍 관계 업데이트 예시:")
    example_count = 0
    for idx, rel in relationships_df.iterrows():
        if rel['source'] in entity_title_mapping or rel['target'] in entity_title_mapping:
            original_source = rel['source']
            original_target = rel['target']
            new_source = entity_title_mapping.get(original_source, original_source)
            new_target = entity_title_mapping.get(original_target, original_target)
            
            if original_source != new_source or original_target != new_target:
                print(f"  [{rel['human_readable_id']}] {original_source} -> {original_target}")
                print(f"       => {new_source} -> {new_target}")
                example_count += 1
                if example_count >= 3:
                    break

# 관계 저장
# human_readable_id 기준으로 정렬
relationships_merged_df = relationships_merged_df.sort_values("human_readable_id").reset_index(drop=True)

# 관계 저장
relationships_merged_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "relationships_merged.parquet")
relationships_merged_df.to_parquet(relationships_merged_path, index=False)
print(f"\n💾 병합된 관계 저장 완료: {relationships_merged_path}")

print("\n✅ A-6 단계 완료: 관계 업데이트 완료")

## 11. 관계 통합 (B단계)

### 11-1. 병합된 관계 데이터 로드 및 분석 (B-1)

In [ ]:
# B-1. 병합된 관계 데이터 로드 및 분석
print("="*60)
print("B-1. 병합된 관계 데이터 로드 및 분석 시작")
print("="*60)

# 병합된 관계 파일 로드
relationships_merged_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "relationships_merged.parquet")
relationships_merged_df = pd.read_parquet(relationships_merged_path)

print(f"✅ 병합된 관계 파일 로드 완료")
print(f"  - 경로: {relationships_merged_path}")
print(f"  - 총 관계 수: {len(relationships_merged_df)}")
print(f"  - 컬럼: {relationships_merged_df.columns.tolist()}")

# 기본 통계
print("📊 기본 통계:")
print(f"  - 유니크한 source 수: {relationships_merged_df['source'].nunique()}")
print(f"  - 유니크한 target 수: {relationships_merged_df['target'].nunique()}")
print(f"  - weight 분포:")
print(f"    • 최소: {relationships_merged_df['weight'].min()}")
print(f"    • 최대: {relationships_merged_df['weight'].max()}")
print(f"    • 평균: {relationships_merged_df['weight'].mean():.2f}")
print(f"    • 중앙값: {relationships_merged_df['weight'].median()}")

# 동일 source-target 쌍 확인
print("🔍 동일 source-target 쌍 확인:")
duplicate_pairs = relationships_merged_df.groupby(['source', 'target']).size()
duplicates = duplicate_pairs[duplicate_pairs > 1]

if len(duplicates) > 0:
    print(f"  ⚠️ 중복된 source-target 쌍 발견: {len(duplicates)}개")
    print("  중복 예시 (최대 5개):")
    for i, ((source, target), count) in enumerate(duplicates.head().items()):
        print(f"    [{i+1}] {source} → {target}: {count}개")
        # 해당 관계들의 description 보기
        dup_rels = relationships_merged_df[(relationships_merged_df['source'] == source) & 
                                          (relationships_merged_df['target'] == target)]
        for j, (_, rel) in enumerate(dup_rels.iterrows()):
            desc_preview = rel['description'][:80] + "..." if len(rel['description']) > 80 else rel['description']
            print(f"        - [{rel['human_readable_id']}] {desc_preview}")
else:
    print("  ✅ 동일한 source-target 쌍이 없습니다 (A-6에서 이미 처리됨)")

# 유사한 관계 가능성 탐색 (source와 target이 반대인 경우)
print("🔄 양방향 관계 확인 (A→B와 B→A):")
bidirectional_count = 0
bidirectional_examples = []

for idx, rel in relationships_merged_df.iterrows():
    # 반대 방향 관계 찾기
    reverse_rel = relationships_merged_df[
        (relationships_merged_df['source'] == rel['target']) & 
        (relationships_merged_df['target'] == rel['source'])
    ]
    
    if len(reverse_rel) > 0 and idx < reverse_rel.index[0]:  # 중복 카운트 방지
        bidirectional_count += 1
        if len(bidirectional_examples) < 3:
            bidirectional_examples.append({
                'forward': rel,
                'reverse': reverse_rel.iloc[0]
            })

print(f"  - 양방향 관계 쌍: {bidirectional_count}개")
if bidirectional_examples:
    print("  양방향 관계 예시:")
    for i, example in enumerate(bidirectional_examples):
        fwd = example['forward']
        rev = example['reverse']
        print(f"[{i+1}] 관계 쌍:")
        print(f"      → [{fwd['human_readable_id']}] {fwd['source']} → {fwd['target']}")
        print(f"        {fwd['description'][:60]}...")
        print(f"      ← [{rev['human_readable_id']}] {rev['source']} → {rev['target']}")
        print(f"        {rev['description'][:60]}...")

# 관계 설명 길이 분석
print("📝 Description 분석:")
desc_lengths = relationships_merged_df['description'].str.len()
print(f"  - 평균 길이: {desc_lengths.mean():.0f} 문자")
print(f"  - 최소 길이: {desc_lengths.min()} 문자")
print(f"  - 최대 길이: {desc_lengths.max()} 문자")

# || 구분자로 병합된 description 확인
merged_descriptions = relationships_merged_df[relationships_merged_df['description'].str.contains(' \|\| ', regex=False)]
print(f"  - '||'로 병합된 description: {len(merged_descriptions)}개")
if len(merged_descriptions) > 0:
    print("    (A-6에서 이미 일부 관계가 병합됨)")

print("✅ B-1 단계 완료: 병합된 관계 데이터 분석 완료")

### 11-2. 관계 텍스트 임베딩 생성 (B-2)

In [ ]:
# B-2. 관계 텍스트 임베딩 생성
print("="*60)
print("B-2. 관계 텍스트 임베딩 생성 시작")
print("="*60)

# 관계 텍스트 생성 전략
# A-2와 유사하게 간단한 형태로: "source -> target | description"
print("\n🔤 관계 텍스트 생성 중...")
relationship_texts = relationships_merged_df.apply(
    lambda row: f"{row['source']} -> {row['target']} | {row['description'][:200]}", 
    axis=1
).tolist()

# 몇 가지 샘플 출력
print("\n📝 관계 텍스트 샘플 (처음 5개):")
for i in range(min(5, len(relationship_texts))):
    text_preview = relationship_texts[i][:150] + "..." if len(relationship_texts[i]) > 150 else relationship_texts[i]
    print(f"  [{i}] {text_preview}")

# 임베딩 생성
print(f"\n🧮 {len(relationship_texts)}개 관계에 대한 임베딩 생성 중...")
print(f"   (임베딩 모델: {snu_kg_config['kggen']['embedding_model']})")

# 이미 로드된 모델 사용 (A-2에서 로드됨)
relationship_embeddings = model.encode(relationship_texts, show_progress_bar=True)

print(f"\n✅ 임베딩 생성 완료!")
print(f"  - 임베딩 shape: {relationship_embeddings.shape}")
print(f"  - 임베딩 차원: {relationship_embeddings.shape[1]}")

# 임베딩 저장
relationship_embeddings_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "relationship_embeddings.npy")
np.save(relationship_embeddings_path, relationship_embeddings)
print(f"\n💾 관계 임베딩 저장 완료: {relationship_embeddings_path}")

# 임베딩 품질 체크 (샘플로 몇 개의 유사도 계산)
print("\n🔍 임베딩 품질 체크 (유사한 관계 찾기):")
from sklearn.metrics.pairwise import cosine_similarity

# 첫 번째 관계와 다른 관계들의 유사도 계산
if len(relationship_embeddings) > 10:
    sample_idx = 0
    similarities = cosine_similarity([relationship_embeddings[sample_idx]], relationship_embeddings)[0]
    
    # 자기 자신 제외하고 가장 유사한 5개 찾기
    similar_indices = np.argsort(similarities)[::-1][1:6]
    
    print(f"\n  기준 관계 [{sample_idx}]: {relationship_texts[sample_idx][:100]}...")
    print("\n  가장 유사한 관계들:")
    for rank, idx in enumerate(similar_indices, 1):
        sim_score = similarities[idx]
        print(f"    {rank}. [{idx}] (유사도: {sim_score:.3f})")
        print(f"       {relationship_texts[idx][:100]}...")

print("\n✅ B-2 단계 완료: 관계 텍스트 임베딩 생성 완료")

### 11-3. 관계 클러스터링 (B-3)

In [ ]:
# B-3. 관계 클러스터링
print("="*60)
print("B-3. 관계 클러스터링 시작")
print("="*60)

# 관계에 대한 클러스터링 설정
print("\n🎯 클러스터링 설정:")
rel_cluster_size = 30  # 원본은 128개 size로 이용함
rel_n_clusters = max(1, len(relationships_merged_df) // rel_cluster_size)
print(f"  - 총 관계 수: {len(relationships_merged_df)}")
print(f"  - 클러스터 크기 기준: {rel_cluster_size}")
print(f"  - 클러스터 수: {rel_n_clusters}")

# 클러스터 수가 너무 많으면 조정
if rel_n_clusters > 30:
    rel_n_clusters = 30
    print(f"  - 조정된 클러스터 수: {rel_n_clusters} (최대값으로 제한)")

# K-means 실행
if rel_n_clusters == 1:
    print("\n⚠️ 관계 수가 적어 모든 관계를 하나의 클러스터로 처리합니다.")
    rel_clusters = np.zeros(len(relationships_merged_df), dtype=int)
else:
    print(f"\n🔄 K-means 클러스터링 실행 중 (k={rel_n_clusters})...")
    rel_kmeans = KMeans(
        n_clusters=rel_n_clusters, 
        random_state=42,
        n_init=10,
        max_iter=300
    )
    rel_clusters = rel_kmeans.fit_predict(relationship_embeddings)
    
    # Silhouette score 계산
    if rel_n_clusters > 1:
        rel_silhouette_avg = silhouette_score(relationship_embeddings, rel_clusters)
        print(f"\n📊 클러스터링 품질:")
        print(f"  - Silhouette Score: {rel_silhouette_avg:.3f}")

# 클러스터별 관계 그룹화
rel_cluster_groups = {}
for idx, cluster_id in enumerate(rel_clusters):
    if cluster_id not in rel_cluster_groups:
        rel_cluster_groups[cluster_id] = []
    rel_cluster_groups[cluster_id].append(idx)

# 클러스터 통계
print(f"\n📈 클러스터링 결과:")
print(f"  - 생성된 클러스터 수: {len(rel_cluster_groups)}")

rel_cluster_sizes = []
for cluster_id, rel_indices in sorted(rel_cluster_groups.items()):
    size = len(rel_indices)
    rel_cluster_sizes.append(size)

print(f"  - 클러스터 크기 통계:")
print(f"    • 최소: {min(rel_cluster_sizes)}개")
print(f"    • 최대: {max(rel_cluster_sizes)}개")
print(f"    • 평균: {np.mean(rel_cluster_sizes):.1f}개")
print(f"    • 중앙값: {np.median(rel_cluster_sizes):.0f}개")

# 각 클러스터 내의 관계 예시
print("\n🔍 클러스터별 관계 예시 (처음 3개 클러스터):")
for cluster_id in range(min(3, len(rel_cluster_groups))):
    rel_indices = rel_cluster_groups[cluster_id]
    print(f"\n  클러스터 {cluster_id} ({len(rel_indices)}개 관계):")
    
    # 클러스터 내 처음 3개 관계 표시
    for i in range(min(3, len(rel_indices))):
        idx = rel_indices[i]
        rel = relationships_merged_df.iloc[idx]
        desc_preview = rel['description'][:60] + "..." if len(rel['description']) > 60 else rel['description']
        print(f"    - [{rel['human_readable_id']}] {rel['source']} → {rel['target']}")
        print(f"      {desc_preview}")
    
    if len(rel_indices) > 3:
        print(f"    - ... 외 {len(rel_indices) - 3}개")

# 클러스터링 정보 저장
rel_cluster_info = {
    'clusters': rel_clusters,
    'cluster_groups': rel_cluster_groups,
    'n_clusters': rel_n_clusters,
    'cluster_sizes': rel_cluster_sizes
}

rel_cluster_info_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "rel_cluster_info.pkl")
with open(rel_cluster_info_path, 'wb') as f:
    pickle.dump(rel_cluster_info, f)
print(f"\n💾 관계 클러스터 정보 저장 완료: {rel_cluster_info_path}")

print("\n✅ B-3 단계 완료: 관계 클러스터링 완료")

### 11-4. DSPy 기반 관계 중복 제거 (B-4)

In [ ]:
# B-4. DSPy 기반 관계 중복 제거
print("="*60)
print("B-4. DSPy 기반 관계 중복 제거 시작")
print("="*60)

# 프롬프트 파일에서 관계 병합 프롬프트 로드
prompt_file = "<PROJECT_ROOT>/kg_gen/kg-gen_prompt.txt"
if os.path.exists(prompt_file):
    print(f"\n📄 프롬프트 파일 로드: {prompt_file}")
    with open(prompt_file, 'r', encoding='utf-8') as f:
        prompt_content = f.read()
    # 관계 병합 프롬프트 추출 (수정된 라인 번호)
    rel_merge_prompt = prompt_content.split('\n')[126:283]  # 0-based index
    rel_merge_prompt_text = '\n'.join(rel_merge_prompt).strip()
    print("\n📋 관계 병합 프롬프트 로드 완료")
    print(f"   프롬프트 길이: {len(rel_merge_prompt_text)} 문자")
else:
    print(f"\n⚠️ 프롬프트 파일을 찾을 수 없습니다. 기본 프롬프트 사용")
    rel_merge_prompt_text = """한국 농업 문서의 관계 중복 제거를 위한 프롬프트입니다.
    동일하거나 매우 유사한 의미의 관계를 찾아 병합합니다."""

# 관계 병합을 위한 DSPy Signature 정의 (수정: duplicates -> merge_candidates)
class RelationshipDeduplicate(dspy.Signature):
    __doc__ = rel_merge_prompt_text
    
    item: str = dspy.InputField(desc="병합 여부를 확인할 대상 관계 (source -> target: description)")
    candidates: list[str] = dspy.InputField(desc="클러스터 내 다른 관계들")
    merge_candidates: list[str] = dspy.OutputField(desc="item과 병합해야 할 관계 리스트 (병합이 적절한 경우만)")
    reason: str = dspy.OutputField(desc="병합 결정 이유")

# 관계 병합 그룹을 저장할 리스트
rel_merge_groups = []
rel_processed_indices = set()

# 병합 이유와 상세 정보를 저장할 리스트
rel_merge_reasons = []

print("\n🔍 클러스터별 관계 중복 제거 시작...")
total_rel_clusters = len(rel_cluster_groups)

# 진행 상황 추적 변수
total_rel_comparisons = 0
total_rel_api_calls = 0

# 각 클러스터별로 중복 제거
for cluster_idx, (cluster_id, rel_indices) in enumerate(rel_cluster_groups.items()):
    if len(rel_indices) <= 1:
        continue  # 1개 이하면 병합할 게 없음
    
    print(f"\n  클러스터 {cluster_id} 처리 중... ({cluster_idx + 1}/{total_rel_clusters})")
    print(f"    - 관계 수: {len(rel_indices)}개")
    
    # 클러스터 내 관계 정보 준비
    cluster_relations = []
    for idx in rel_indices:
        rel = relationships_merged_df.iloc[idx]
        # 관계 정보를 문자열로 표현 (source -> target: description)
        rel_info = f"{rel['source']} -> {rel['target']}: {rel['description'][:100]}"
        cluster_relations.append((idx, rel_info))
    
    # 이미 처리된 관계는 제외
    unprocessed_relations = [
        (idx, info) for idx, info in cluster_relations 
        if idx not in rel_processed_indices
    ]
    
    # 클러스터 내에서 중복 찾기
    cluster_rel_merge_groups = []
    cluster_comparisons = 0
    
    for i, (current_idx, current_info) in enumerate(unprocessed_relations):
        if current_idx in rel_processed_indices:
            continue
            
        # 나머지 관계들과 비교
        other_relations = [info for idx, info in unprocessed_relations[i+1:] 
                          if idx not in rel_processed_indices]
        
        if not other_relations:
            continue
        
        cluster_comparisons += 1
        total_rel_comparisons += 1
        
        try:
            # DSPy를 사용하여 병합 대상 찾기
            deduplicate = dspy.Predict(RelationshipDeduplicate)
            result = deduplicate(
                item=current_info,
                candidates=other_relations
            )
            total_rel_api_calls += 1
            
            # 병합할 관계가 있으면 (merge_candidates로 변경)
            if result.merge_candidates and len(result.merge_candidates) > 0:
                # 병합 대상 관계의 인덱스 찾기
                merge_indices = []
                for merge_candidate in result.merge_candidates:
                    for idx, info in unprocessed_relations[i+1:]:
                        if info == merge_candidate and idx not in rel_processed_indices:
                            merge_indices.append(idx)
                            rel_processed_indices.add(idx)
                            break
                
                if merge_indices:
                    # 현재 관계도 처리됨으로 표시
                    rel_processed_indices.add(current_idx)
                    
                    # human_readable_id가 가장 낮은 관계를 대표로
                    all_indices = [current_idx] + merge_indices
                    representative_idx = min(all_indices, 
                                           key=lambda x: relationships_merged_df.iloc[x]['human_readable_id'])
                    
                    rel_merge_groups.append((representative_idx, all_indices))
                    cluster_rel_merge_groups.append((representative_idx, all_indices))
                    
                    # 병합 상세 정보 수집 (NumPy 타입을 Python 네이티브 타입으로 변환)
                    rel_merge_detail = {
                        'group_id': len(rel_merge_groups) - 1,
                        'representative': {
                            'index': int(representative_idx),
                            'human_readable_id': int(relationships_merged_df.iloc[representative_idx]['human_readable_id']),
                            'source': str(relationships_merged_df.iloc[representative_idx]['source']),
                            'target': str(relationships_merged_df.iloc[representative_idx]['target']),
                            'description': str(relationships_merged_df.iloc[representative_idx]['description']),
                            'weight': float(relationships_merged_df.iloc[representative_idx]['weight'])
                        },
                        'merged_relationships': [],
                        'reason': str(result.reason),
                        'cluster_id': int(cluster_id),
                        'timestamp': pd.Timestamp.now().isoformat()
                    }
                    
                    # 병합될 관계들의 상세 정보
                    for idx in all_indices:
                        rel_detail = {
                            'index': int(idx),
                            'human_readable_id': int(relationships_merged_df.iloc[idx]['human_readable_id']),
                            'source': str(relationships_merged_df.iloc[idx]['source']),
                            'target': str(relationships_merged_df.iloc[idx]['target']),
                            'description': str(relationships_merged_df.iloc[idx]['description']),
                            'weight': float(relationships_merged_df.iloc[idx]['weight']),
                            'combined_degree': float(relationships_merged_df.iloc[idx]['combined_degree']),
                            'text_unit_ids': relationships_merged_df.iloc[idx]['text_unit_ids'].tolist() if hasattr(relationships_merged_df.iloc[idx]['text_unit_ids'], 'tolist') else list(relationships_merged_df.iloc[idx]['text_unit_ids'])
                        }
                        rel_merge_detail['merged_relationships'].append(rel_detail)
                    
                    rel_merge_reasons.append(rel_merge_detail)
                    
                    # 병합 정보 출력
                    rep_rel = relationships_merged_df.iloc[representative_idx]
                    print(f"\n    ✅ 관계 병합 그룹 발견:")
                    print(f"       대표: [{rep_rel['human_readable_id']}] {rep_rel['source']} → {rep_rel['target']}")
                    print(f"       병합 대상 ({len(all_indices)}개):")
                    for idx in all_indices:
                        if idx != representative_idx:
                            rel = relationships_merged_df.iloc[idx]
                            print(f"         - [{rel['human_readable_id']}] {rel['source']} → {rel['target']}")
                            print(f"           {rel['description'][:60]}...")
                    print(f"       이유: {result.reason[:100]}...")
                    
        except Exception as e:
            print(f"    ⚠️ DSPy 처리 중 오류: {e}")
            continue
    
    if cluster_rel_merge_groups:
        print(f"    → 클러스터 {cluster_id}에서 {len(cluster_rel_merge_groups)}개 관계 병합 그룹 생성")
    else:
        print(f"    → 클러스터 {cluster_id}에서 병합할 관계 없음")
    
    print(f"    → 클러스터 내 비교 수: {cluster_comparisons}")

# 병합 통계
print(f"\n📊 관계 중복 제거 결과:")
print(f"  - 처리한 클러스터 수: {total_rel_clusters}개")
print(f"  - 총 비교 수: {total_rel_comparisons}")
print(f"  - API 호출 수: {total_rel_api_calls}")
print(f"  - 총 관계 병합 그룹 수: {len(rel_merge_groups)}")

total_relations_to_merge = sum(len(indices) for _, indices in rel_merge_groups)
print(f"  - 병합될 관계 수: {total_relations_to_merge}")
if total_relations_to_merge > 0:
    print(f"  - 병합 후 관계 수: {len(relationships_merged_df) - total_relations_to_merge + len(rel_merge_groups)}")
    print(f"  - 감소율: {((total_relations_to_merge - len(rel_merge_groups)) / len(relationships_merged_df) * 100):.1f}%")
else:
    print(f"  - 병합 후 관계 수: {len(relationships_merged_df)} (변화 없음)")

# 관계 병합 정보 저장
rel_merge_info = {
    'merge_groups': rel_merge_groups,
    'processed_indices': rel_processed_indices,
    'total_relations': len(relationships_merged_df),
    'relations_after_merge': len(relationships_merged_df) - total_relations_to_merge + len(rel_merge_groups) if total_relations_to_merge > 0 else len(relationships_merged_df),
    'total_comparisons': total_rel_comparisons,
    'total_api_calls': total_rel_api_calls
}

rel_merge_info_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "rel_merge_info.pkl")
with open(rel_merge_info_path, 'wb') as f:
    pickle.dump(rel_merge_info, f)
print(f"\n💾 관계 병합 정보 저장 완료: {rel_merge_info_path}")

# 병합 이유와 상세 정보 저장
rel_merge_reasons_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "merged_relationships_reasons.json")
with open(rel_merge_reasons_path, 'w', encoding='utf-8') as f:
    json.dump(rel_merge_reasons, f, ensure_ascii=False, indent=2)
print(f"💾 관계 병합 이유 및 상세 정보 저장 완료: {rel_merge_reasons_path}")
print(f"   - 저장된 관계 병합 그룹 수: {len(rel_merge_reasons)}개")

# 병합 이유 요약 출력
if rel_merge_reasons:
    print("\n📋 관계 병합 이유 요약:")
    for i, reason in enumerate(rel_merge_reasons[:3]):  # 처음 3개만 출력
        print(f"\n  [{i+1}] {reason['representative']['source']} → {reason['representative']['target']} 그룹")
        print(f"      병합 관계 수: {len(reason['merged_relationships'])}개")
        print(f"      이유: {reason['reason'][:150]}...")
    if len(rel_merge_reasons) > 3:
        print(f"\n  ... 외 {len(rel_merge_reasons) - 3}개 그룹")

print("\n✅ B-4 단계 완료: DSPy 기반 관계 중복 제거 완료")

### B-*. 양방향 관계만 DSPy로 중복 제거
해당 과정은 B-2, B-3, B-4의 과정을 대체합니다.

In [ ]:
# %% [markdown]
# ### B*. 양방향 관계만 대상으로 한 DSPy 기반 관계 중복 제거
# 
# B-2, B-3, B-4 과정을 대체하여 양방향 관계(A→B, B→A)만을 찾아 중복 제거

# %%
# DSPy 설정 (중요: B* 실행 전에 반드시 설정해야 함)
print("\n🔧 DSPy 설정 중...")
import dspy
from pathlib import Path
import pickle
import json

# DSPy LM 설정
lm = dspy.LM(
    model=f"openai/{snu_kg_config['graphrag']['model']}",
    temperature=snu_kg_config['kggen']['dedup_temperature'],
    max_tokens=1000
)
dspy.configure(lm=lm)
print(f"✅ DSPy 설정 완료: {snu_kg_config['graphrag']['model']}")

# 프롬프트 파일에서 관계 병합 프롬프트 로드
prompt_file = "<PROJECT_ROOT>/kg_gen/kg-gen_prompt.txt"
if os.path.exists(prompt_file):
    print(f"\n📄 프롬프트 파일 로드: {prompt_file}")
    with open(prompt_file, 'r', encoding='utf-8') as f:
        prompt_content = f.read()
    # 관계 병합 프롬프트 추출 (수정된 라인 번호)
    rel_merge_prompt = prompt_content.split('\n')[126:283]  # 0-based index
    rel_merge_prompt_text = '\n'.join(rel_merge_prompt).strip()
    print("\n📋 관계 병합 프롬프트 로드 완료")
    print(f"   프롬프트 길이: {len(rel_merge_prompt_text)} 문자")
else:
    print(f"\n⚠️ 프롬프트 파일을 찾을 수 없습니다. 기본 프롬프트 사용")
    rel_merge_prompt_text = """한국 농업 문서의 관계 중복 제거를 위한 프롬프트입니다.
    동일하거나 매우 유사한 의미의 관계를 찾아 병합합니다."""

# DSPy Signature 정의 (B-4와 동일한 구조)
class RelationshipDeduplicate(dspy.Signature):
    __doc__ = rel_merge_prompt_text
    
    item: str = dspy.InputField(desc="병합 여부를 확인할 대상 관계 (source → target: description)")
    candidates: list[str] = dspy.InputField(desc="클러스터 내 다른 관계들")
    merge_candidates: list[str] = dspy.OutputField(desc="item과 병합해야 할 관계 리스트 (병합이 적절한 경우만)")
    reason: str = dspy.OutputField(desc="병합 결정 이유")

# GraphRAG 관계 로드 (A-6에서 생성된 병합된 관계 사용)
graphrag_relationships_path = Path("<PROJECT_ROOT>/kg_gen/snu_kg_output/intermediate/relationships_merged.parquet")
if not graphrag_relationships_path.exists():
    print(f"❌ 파일을 찾을 수 없습니다: {graphrag_relationships_path}")
    raise FileNotFoundError(f"File not found: {graphrag_relationships_path}")

rel_df = pd.read_parquet(graphrag_relationships_path)
print(f"\n📊 로드된 관계: {len(rel_df):,}개")

# 양방향 관계 찾기
print("\n🔍 양방향 관계 찾는 중...")
bidirectional_pairs = []
relationship_dict = {}

# 관계를 딕셔너리로 정리 (빠른 조회를 위해)
for idx, row in rel_df.iterrows():
    key = (row['source'], row['target'])
    if key not in relationship_dict:
        relationship_dict[key] = []
    relationship_dict[key].append({
        'index': idx,
        'description': row['description'],
        'weight': row.get('weight', 1.0),
        'human_readable_id': row['human_readable_id'],
        'id': row['id'],
        'combined_degree': row.get('combined_degree', 0),
        'text_unit_ids': row.get('text_unit_ids', [])
    })

# 양방향 쌍 찾기
processed = set()
for (source, target), relations in relationship_dict.items():
    if (source, target) in processed or (target, source) in processed:
        continue
    
    # 역방향 관계 확인
    reverse_key = (target, source)
    if reverse_key in relationship_dict:
        # 양방향 관계 발견
        bidirectional_pairs.append({
            'forward': {'source': source, 'target': target, 'relations': relations},
            'reverse': {'source': target, 'target': source, 'relations': relationship_dict[reverse_key]}
        })
        processed.add((source, target))
        processed.add((target, source))

print(f"✅ 발견된 양방향 관계 쌍: {len(bidirectional_pairs)}개")

# 양방향 관계들만 DSPy로 처리
print("\n🤖 DSPy 기반 양방향 관계 중복 제거 시작...")
print(f"   모든 {len(bidirectional_pairs)}개 양방향 관계 쌍을 처리합니다.")

# 병합된 관계 저장
merged_relationships = []
removed_count = 0
processed_rel_indices = set()

# B-5를 위한 병합 정보 저장
rel_merge_groups = []
rel_merge_reasons = []

# DSPy 예측기 생성
deduplicate = dspy.Predict(RelationshipDeduplicate)

for i, pair in enumerate(bidirectional_pairs):
    print(f"\n처리 중... 쌍 {i+1}/{len(bidirectional_pairs)}")
    
    try:
        # 양방향 관계를 하나의 클러스터로 처리
        cluster_relationships = []
        cluster_indices = []
        
        # Forward 관계들
        for rel in pair['forward']['relations']:
            cluster_relationships.append({
                'source': pair['forward']['source'],
                'target': pair['forward']['target'],
                'description': rel['description'],
                'weight': rel['weight'],
                'index': rel['index']
            })
            cluster_indices.append(rel['index'])
            processed_rel_indices.add(rel['index'])
        
        # Reverse 관계들
        for rel in pair['reverse']['relations']:
            cluster_relationships.append({
                'source': pair['reverse']['source'],
                'target': pair['reverse']['target'],
                'description': rel['description'],
                'weight': rel['weight'],
                'index': rel['index']
            })
            cluster_indices.append(rel['index'])
            processed_rel_indices.add(rel['index'])
        
        # 클러스터 내에서 중복 제거
        if len(cluster_relationships) > 1:
            # 첫 번째 관계를 기준으로
            base_rel = cluster_relationships[0]
            base_str = f"{base_rel['source']} → {base_rel['target']}: {base_rel['description'][:100]}"
            
            # 나머지 관계들
            candidate_strs = []
            for rel in cluster_relationships[1:]:
                rel_str = f"{rel['source']} → {rel['target']}: {rel['description'][:100]}"
                candidate_strs.append(rel_str)
            
            # DSPy로 병합 대상 찾기
            result = deduplicate(
                item=base_str,
                candidates=candidate_strs
            )
            
            # 병합할 관계 인덱스 찾기
            merge_indices = [0]  # 기준 관계 포함
            if result.merge_candidates:
                for j, cand_str in enumerate(candidate_strs):
                    if cand_str in result.merge_candidates:
                        merge_indices.append(j + 1)
            
            # 병합 수행
            if len(merge_indices) > 1:
                # 대표 관계 인덱스
                representative_idx = cluster_indices[merge_indices[0]]
                merge_group_indices = [cluster_indices[idx] for idx in merge_indices]
                
                # 병합 그룹 저장 (B-5용)
                rel_merge_groups.append((representative_idx, merge_group_indices))
                
                # 병합된 관계 생성
                merged_rel = rel_df.iloc[representative_idx].to_dict()
                
                # weight 합산
                total_weight = sum(cluster_relationships[idx]['weight'] for idx in merge_indices)
                merged_rel['weight'] = total_weight
                
                # description 병합
                descriptions = []
                for idx in merge_indices:
                    desc = cluster_relationships[idx]['description']
                    if desc and desc not in descriptions:
                        descriptions.append(desc)
                merged_rel['description'] = ' || '.join(descriptions)
                
                merged_relationships.append(merged_rel)
                removed_count += len(merge_indices) - 1
                
                # 병합 이유 저장 (B-5용)
                rel_merge_detail = {
                    'group_id': len(rel_merge_groups) - 1,
                    'representative': {
                        'index': int(representative_idx),
                        'human_readable_id': int(rel_df.iloc[representative_idx]['human_readable_id']),
                        'source': str(rel_df.iloc[representative_idx]['source']),
                        'target': str(rel_df.iloc[representative_idx]['target']),
                        'description': str(rel_df.iloc[representative_idx]['description']),
                        'weight': float(total_weight)
                    },
                    'merged_relationships': [],
                    'reason': str(result.reason),
                    'cluster_id': f"bidirectional_pair_{i}",
                    'timestamp': pd.Timestamp.now().isoformat()
                }
                
                # 병합된 관계들의 상세 정보
                for merge_idx in merge_indices:
                    rel_idx = cluster_indices[merge_idx]
                    rel_detail = {
                        'index': int(rel_idx),
                        'human_readable_id': int(rel_df.iloc[rel_idx]['human_readable_id']),
                        'source': str(rel_df.iloc[rel_idx]['source']),
                        'target': str(rel_df.iloc[rel_idx]['target']),
                        'description': str(rel_df.iloc[rel_idx]['description']),
                        'weight': float(rel_df.iloc[rel_idx]['weight']),
                        'combined_degree': float(rel_df.iloc[rel_idx].get('combined_degree', 0)),
                        'text_unit_ids': rel_df.iloc[rel_idx]['text_unit_ids'].tolist() if hasattr(rel_df.iloc[rel_idx]['text_unit_ids'], 'tolist') else list(rel_df.iloc[rel_idx]['text_unit_ids'])
                    }
                    rel_merge_detail['merged_relationships'].append(rel_detail)
                
                rel_merge_reasons.append(rel_merge_detail)
                
                # 병합되지 않은 관계들 추가
                for j, rel_idx in enumerate(cluster_indices):
                    if j not in merge_indices:
                        merged_relationships.append(rel_df.iloc[rel_idx].to_dict())
                
                print(f"  - 원본 관계: {len(cluster_relationships)}개")
                print(f"  - 병합됨: {len(merge_indices)}개 → 1개")
                print(f"  - 제거됨: {len(merge_indices) - 1}개")
            else:
                # 병합할 것이 없으면 모든 관계 추가
                for rel_idx in cluster_indices:
                    merged_relationships.append(rel_df.iloc[rel_idx].to_dict())
        else:
            # 관계가 하나뿐이면 그대로 추가
            for rel_idx in cluster_indices:
                merged_relationships.append(rel_df.iloc[rel_idx].to_dict())
    
    except Exception as e:
        print(f"  ⚠️ DSPy 처리 중 오류 (쌍 {i+1}): {str(e)}")
        # 오류 발생 시 원본 관계 유지
        for rel_idx in cluster_indices:
            merged_relationships.append(rel_df.iloc[rel_idx].to_dict())

# 양방향이 아닌 관계들 추가 (수정하지 않음)
print("\n📋 단방향 관계 추가 중...")
unidirectional_count = 0
for idx, row in rel_df.iterrows():
    if idx not in processed_rel_indices:
        merged_relationships.append(row.to_dict())
        unidirectional_count += 1

print(f"✅ 추가된 단방향 관계: {unidirectional_count}개")

# 결과 DataFrame 생성
print("\n📊 최종 결과 생성 중...")
merged_rel_df = pd.DataFrame(merged_relationships)

# human_readable_id 재정렬
merged_rel_df = merged_rel_df.sort_values('human_readable_id').reset_index(drop=True)

# 필요한 컬럼 순서 맞추기
required_columns = ['id', 'human_readable_id', 'source', 'target', 'description', 'weight', 'combined_degree', 'text_unit_ids']
merged_rel_df = merged_rel_df[required_columns]

# 통계 출력
print("\n📈 B* 중복 제거 결과:")
print(f"  • 원본 관계: {len(rel_df):,}개")
print(f"  • 양방향 관계 쌍: {len(bidirectional_pairs)}개")
print(f"  • 처리된 양방향 관계: {len(bidirectional_pairs) * 2}개")
print(f"  • 병합 후 관계: {len(merged_rel_df):,}개")
print(f"  • 제거된 관계: {removed_count}개")
if removed_count > 0:
    print(f"  • 감소율: {(removed_count/len(rel_df)*100):.1f}%")

# B-5를 위한 병합 정보 저장
rel_merge_info = {
    'merge_groups': rel_merge_groups,
    'processed_indices': processed_rel_indices,
    'total_relations': len(rel_df),
    'relations_after_merge': len(merged_rel_df),
    'total_comparisons': len(bidirectional_pairs),  # 양방향 쌍 수
    'total_api_calls': len(bidirectional_pairs)     # DSPy 호출 수
}

# pickle 파일 저장
rel_merge_info_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "rel_merge_info.pkl")
with open(rel_merge_info_path, 'wb') as f:
    pickle.dump(rel_merge_info, f)
print(f"\n💾 관계 병합 정보 저장 완료: {rel_merge_info_path}")

# JSON 파일 저장
rel_merge_reasons_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "merged_relationships_reasons.json")
with open(rel_merge_reasons_path, 'w', encoding='utf-8') as f:
    json.dump(rel_merge_reasons, f, ensure_ascii=False, indent=2)
print(f"💾 관계 병합 이유 저장 완료: {rel_merge_reasons_path}")

# 병합된 관계 파일 저장 (B-5가 읽을 위치)
relationships_final_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "relationships_final.parquet")
merged_rel_df.to_parquet(relationships_final_path, index=False)
print(f"💾 최종 관계 저장 완료: {relationships_final_path}")

print("\n✅ B* 완료: B-5와 호환되는 모든 파일 생성 완료")
print("   이제 B-5를 실행하여 최종 정리를 진행할 수 있습니다.")

### 11-5. 관계 병합 및 최종 정리 (B-5)

In [ ]:
# B-5. 관계 병합 및 최종 정리
print("="*60)
print("B-5. 관계 병합 및 최종 정리 시작")
print("="*60)

# 관계 병합 정보 로드
with open(rel_merge_info_path, 'rb') as f:
    rel_merge_info = pickle.load(f)

rel_merge_groups = rel_merge_info['merge_groups']
print(f"\n📊 병합할 관계 그룹 수: {len(rel_merge_groups)}")

# 병합된 관계 이유 정보도 로드 (참조용)
if os.path.exists(rel_merge_reasons_path):
    with open(rel_merge_reasons_path, 'r', encoding='utf-8') as f:
        saved_rel_merge_reasons = json.load(f)
    print(f"📋 저장된 병합 이유 정보: {len(saved_rel_merge_reasons)}개 그룹")

# 새로운 관계 데이터프레임 준비
merged_relations = []
processed_rel_indices = set()

# 병합 그룹별로 처리
print("\n🔄 관계 병합 처리 중...")
for group_idx, (representative_idx, indices) in enumerate(rel_merge_groups):
    print(f"\n  그룹 {group_idx + 1}/{len(rel_merge_groups)} 처리 중...")
    
    # 병합될 관계들 가져오기
    relations_to_merge = [relationships_merged_df.iloc[idx] for idx in indices]
    
    # 대표 관계 (human_readable_id가 가장 낮은 것)
    representative = relationships_merged_df.iloc[representative_idx]
    
    # 병합된 관계 생성
    merged_rel = {
        'id': representative['id'],  # 대표 관계의 ID 사용
        'human_readable_id': representative['human_readable_id'],  # 가장 낮은 human_readable_id
        'source': representative['source'],  # 대표 관계의 source
        'target': representative['target'],  # 대표 관계의 target
        'description': '',  # 모든 description 병합 (아래에서 처리)
        'weight': 0.0,  # 모든 weight 합산 (아래에서 처리)
        'combined_degree': representative['combined_degree'],  # 일단 대표값 사용 (C단계에서 재계산)
        'text_unit_ids': []  # 모든 text_unit_ids 합집합 (아래에서 처리)
    }
    
    # Description 병합 (모든 원본 정보 보존)
    descriptions = []
    rel_num = f"{len(relations_to_merge)}개의 관계가 병합된 정보입니다."
    descriptions.append(rel_num)
    
    for i, rel in enumerate(relations_to_merge, 1):
        desc = rel['description']
        if desc and str(desc).strip() and str(desc) != 'nan':
            # 원본 관계 정보 포함
            rel_info = f"[{i}번 관계: {rel['source']} → {rel['target']} (text_unit_ids: {rel['text_unit_ids']}) {desc}]"
            descriptions.append(rel_info)
    
    # 중복 제거하면서 모든 description 병합
    unique_descriptions = list(dict.fromkeys(descriptions))  # 순서 유지하며 중복 제거
    merged_rel['description'] = ' || '.join(unique_descriptions)
    
    # text_unit_ids 병합 (합집합)
    merged_rel['text_unit_ids'] = merge_text_unit_ids(
        [rel['text_unit_ids'] for rel in relations_to_merge]
    )
    
    # weight 합산
    merged_rel['weight'] = sum(rel['weight'] for rel in relations_to_merge)
    
    # 병합 정보 출력
    print(f"    대표: [{representative['human_readable_id']}] {representative['source']} → {representative['target']}")
    print(f"    병합 관계 수: {len(relations_to_merge)}")
    print(f"    병합 후 weight: {merged_rel['weight']}")
    print(f"    text_unit_ids 수: {len(merged_rel['text_unit_ids'])}")
    
    # 병합 이유 정보가 있으면 출력
    if group_idx < len(saved_rel_merge_reasons):
        reason_info = saved_rel_merge_reasons[group_idx]
        print(f"    병합 이유: {reason_info['reason'][:100]}...")
    
    merged_relations.append(merged_rel)
    processed_rel_indices.update(indices)

# 병합되지 않은 관계들 추가
print("\n📝 병합되지 않은 관계 처리 중...")
unmerged_count = 0
for idx, rel in relationships_merged_df.iterrows():
    if idx not in processed_rel_indices:
        merged_relations.append(rel.to_dict())
        unmerged_count += 1

print(f"  - 병합되지 않은 관계: {unmerged_count}개")

# 새 데이터프레임 생성
relationships_final_df = pd.DataFrame(merged_relations)

# 컬럼 순서 유지
relationships_final_df = relationships_final_df[relationships_merged_df.columns]

# 통계 출력
print(f"\n📊 최종 관계 병합 결과:")
print(f"  - B-4 이전 관계 수: {len(relationships_merged_df)}")
print(f"  - 병합 그룹 수: {len(rel_merge_groups)}")
print(f"  - 최종 관계 수: {len(relationships_final_df)}")
print(f"  - 총 감소율: {((len(relationships_merged_df) - len(relationships_final_df)) / len(relationships_merged_df) * 100):.1f}%")

# 병합 전후 비교 샘플
if len(rel_merge_groups) > 0:
    print("\n🔍 관계 병합 예시 (첫 번째 그룹):")
    first_group_rep_idx, first_group_indices = rel_merge_groups[0]
    print(f"  병합 전:")
    for idx in first_group_indices[:3]:  # 최대 3개만
        rel = relationships_merged_df.iloc[idx]
        print(f"    - [{rel['human_readable_id']}] {rel['source']} → {rel['target']}")
        print(f"      {rel['description'][:60]}...")
    if len(first_group_indices) > 3:
        print(f"    - ... 외 {len(first_group_indices) - 3}개")
    
    # 병합 후 관계 찾기
    merged_rel = relationships_final_df[
        relationships_final_df['human_readable_id'] == relationships_merged_df.iloc[first_group_rep_idx]['human_readable_id']
    ].iloc[0]
    print(f"  병합 후:")
    print(f"    - [{merged_rel['human_readable_id']}] {merged_rel['source']} → {merged_rel['target']}")
    print(f"    - Weight: {merged_rel['weight']}")
    print(f"    - Description 길이: {len(merged_rel['description'])} 문자")

# weight 분포 분석
print("\n📈 Weight 분포 변화:")
print(f"  병합 전:")
print(f"    - 평균: {relationships_merged_df['weight'].mean():.2f}")
print(f"    - 최대: {relationships_merged_df['weight'].max()}")
print(f"  병합 후:")
print(f"    - 평균: {relationships_final_df['weight'].mean():.2f}")
print(f"    - 최대: {relationships_final_df['weight'].max()}")

# 최종 관계 저장
# human_readable_id 기준으로 정렬
relationships_final_df = relationships_final_df.sort_values("human_readable_id").reset_index(drop=True)

# 최종 관계 저장
relationships_final_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "relationships_final.parquet")
relationships_final_df.to_parquet(relationships_final_path, index=False)
print(f"\n💾 최종 관계 파일 저장 완료: {relationships_final_path}")

# 전체 프로세스 요약
print("\n" + "="*60)
print("📊 전체 관계 처리 프로세스 요약:")
print(f"  1. 원본 관계 수: {len(relationships_df)} (from GraphRAG)")
print(f"  2. A-6 후 관계 수: {len(relationships_merged_df)} (엔티티 병합에 따른 업데이트)")
print(f"  3. B-5 후 최종 관계 수: {len(relationships_final_df)} (관계 병합 완료)")
print(f"  4. 총 관계 감소: {len(relationships_df) - len(relationships_final_df)}개 ({((len(relationships_df) - len(relationships_final_df)) / len(relationships_df) * 100):.1f}%)")

print("\n✅ B-5 단계 완료: 관계 병합 및 최종 정리 완료")
print("="*60)

## 12. 정규화 과정 (C단계)

병합된 엔티티와 관계의 degree, weight, combined_degree 값을 재계산하여 그래프의 일관성을 유지합니다.

In [ ]:
# C. 정규화 과정 - degree, weight, combined_degree 재계산 및 ID 정규화
print("="*60)
print("C. 정규화 과정 시작")
print("="*60)

# 최종 파일들 로드
entities_final_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "entities_merged.parquet")
relationships_final_path = os.path.join(SNU_KG_OUTPUT, "intermediate", "relationships_final.parquet")

if not os.path.exists(entities_final_path):
    print("❌ 엔티티 파일을 찾을 수 없습니다.")
    raise FileNotFoundError(f"File not found: {entities_final_path}")

if not os.path.exists(relationships_final_path):
    print("❌ 관계 파일을 찾을 수 없습니다.")
    raise FileNotFoundError(f"File not found: {relationships_final_path}")

# 데이터 로드
entities_final_df = pd.read_parquet(entities_final_path)
relationships_final_df = pd.read_parquet(relationships_final_path)

print(f"✅ 데이터 로드 완료")
print(f"  - 엔티티: {len(entities_final_df)}개")
print(f"  - 관계: {len(relationships_final_df)}개")

# human_readable_id 정규화
print("\n🔢 human_readable_id 정규화 중...")

# 엔티티 ID 정규화
old_entity_ids = entities_final_df['human_readable_id'].tolist()
entities_final_df = entities_final_df.sort_values('human_readable_id').reset_index(drop=True)
entities_final_df['human_readable_id'] = range(len(entities_final_df))
new_entity_ids = entities_final_df['human_readable_id'].tolist()

# ID 변경 매핑 생성 (old_id -> new_id)
entity_id_mapping = dict(zip(old_entity_ids, new_entity_ids))

print(f"  - 엔티티 ID 정규화 완료: 0 ~ {len(entities_final_df)-1}")
print(f"  - ID 변경 예시:")
for i, (old, new) in enumerate(list(entity_id_mapping.items())[:5]):
    if old != new:
        print(f"    • {old} → {new}")
if len([k for k, v in entity_id_mapping.items() if k != v]) > 5:
    print(f"    • ... 외 {len([k for k, v in entity_id_mapping.items() if k != v]) - 5}개 변경")

# 관계 ID 정규화
old_rel_ids = relationships_final_df['human_readable_id'].tolist()
relationships_final_df = relationships_final_df.sort_values('human_readable_id').reset_index(drop=True)
relationships_final_df['human_readable_id'] = range(len(relationships_final_df))
new_rel_ids = relationships_final_df['human_readable_id'].tolist()

print(f"  - 관계 ID 정규화 완료: 0 ~ {len(relationships_final_df)-1}")

# NetworkX 그래프 생성 (degree 계산용)
print("\n🔄 그래프 구조 생성 중...")
G = nx.Graph()

# 노드 추가
for _, entity in entities_final_df.iterrows():
    G.add_node(entity['title'], **entity.to_dict())

# 엣지 추가
for _, rel in relationships_final_df.iterrows():
    # 양방향 그래프이므로 weight만 추가
    if G.has_edge(rel['source'], rel['target']):
        # 이미 엣지가 있으면 weight 누적
        G[rel['source']][rel['target']]['weight'] += rel['weight']
    else:
        # rel.to_dict()에서 weight를 제외한 나머지 속성만 추가
        edge_attrs = rel.to_dict()
        edge_attrs.pop('source', None)  # source와 target은 이미 파라미터로 전달됨
        edge_attrs.pop('target', None)
        # weight는 명시적으로 설정 (중복 방지)
        G.add_edge(rel['source'], rel['target'], **edge_attrs)

print(f"✅ 그래프 생성 완료")
print(f"  - 노드 수: {G.number_of_nodes()}")
print(f"  - 엣지 수: {G.number_of_edges()}")

# 1. 엔티티 degree 재계산
print("\n📊 엔티티 degree 재계산 중...")
degree_dict = dict(G.degree())

# 기존 degree와 비교
old_degrees = entities_final_df['degree'].tolist()
entities_final_df['degree'] = entities_final_df['title'].map(degree_dict).fillna(0).astype(int)
new_degrees = entities_final_df['degree'].tolist()

# degree 변화 분석
degree_changes = sum(1 for old, new in zip(old_degrees, new_degrees) if old != new)
print(f"  - Degree가 변경된 엔티티: {degree_changes}개 ({degree_changes/len(entities_final_df)*100:.1f}%)")
print(f"  - 평균 degree: {entities_final_df['degree'].mean():.2f}")
print(f"  - 최대 degree: {entities_final_df['degree'].max()}")

# Degree 분포 출력
degree_distribution = entities_final_df['degree'].value_counts().sort_index()
print("\n📈 Degree 분포:")
for degree in range(min(5, len(degree_distribution))):
    if degree in degree_distribution:
        count = degree_distribution[degree]
        print(f"  - Degree {degree}: {count}개 엔티티")
if len(degree_distribution) > 5:
    print(f"  - ... (최대 degree: {entities_final_df['degree'].max()})")

# 2. 관계의 combined_degree 재계산
print("\n📊 관계 combined_degree 재계산 중...")

# source와 target의 degree 합
def calculate_combined_degree(row):
    source_degree = degree_dict.get(row['source'], 0)
    target_degree = degree_dict.get(row['target'], 0)
    return source_degree + target_degree

old_combined_degrees = relationships_final_df['combined_degree'].tolist()
relationships_final_df['combined_degree'] = relationships_final_df.apply(calculate_combined_degree, axis=1)
new_combined_degrees = relationships_final_df['combined_degree'].tolist()

# combined_degree 변화 분석
combined_degree_changes = sum(1 for old, new in zip(old_combined_degrees, new_combined_degrees) if old != new)
print(f"  - Combined degree가 변경된 관계: {combined_degree_changes}개 ({combined_degree_changes/len(relationships_final_df)*100:.1f}%)")
print(f"  - 평균 combined degree: {relationships_final_df['combined_degree'].mean():.2f}")
print(f"  - 최대 combined degree: {relationships_final_df['combined_degree'].max()}")

# 3. Weight 정규화 검증
print("\n📊 Weight 분포 분석:")
weight_stats = relationships_final_df['weight'].describe()
print(f"  - 평균: {weight_stats['mean']:.2f}")
print(f"  - 표준편차: {weight_stats['std']:.2f}")
print(f"  - 최소: {weight_stats['min']}")
print(f"  - 최대: {weight_stats['max']}")
print(f"  - 중앙값: {weight_stats['50%']}")

# Weight 분포 히스토그램 데이터
weight_bins = pd.cut(relationships_final_df['weight'], bins=5)
print("\n📊 Weight 구간별 분포:")
for interval, count in weight_bins.value_counts().sort_index().items():
    print(f"  - {interval}: {count}개")

# 4. 고립된 노드 확인
print("\n🔍 고립된 노드 확인:")
isolated_nodes = [node for node in G.nodes() if G.degree(node) == 0]
if isolated_nodes:
    print(f"  ⚠️ 고립된 노드 발견: {len(isolated_nodes)}개")
    for node in isolated_nodes[:5]:
        print(f"    - {node}")
    if len(isolated_nodes) > 5:
        print(f"    - ... 외 {len(isolated_nodes) - 5}개")
else:
    print("  ✅ 고립된 노드 없음")

# 5. 허브 노드 분석 (degree가 높은 노드)
print("\n🌟 허브 노드 (상위 10개):")
top_nodes = entities_final_df.nlargest(10, 'degree')
for idx, node in top_nodes.iterrows():
    print(f"  - [{node['human_readable_id']}] {node['title']} (degree: {node['degree']})")

# 6. ID 연속성 검증
print("\n🔍 ID 연속성 검증:")
entity_ids = sorted(entities_final_df['human_readable_id'].tolist())
rel_ids = sorted(relationships_final_df['human_readable_id'].tolist())

# 엔티티 ID 검증
if entity_ids == list(range(len(entities_final_df))):
    print(f"  ✅ 엔티티 ID 연속성 확인: 0 ~ {len(entities_final_df)-1}")
else:
    print(f"  ❌ 엔티티 ID에 공백 발견")
    
# 관계 ID 검증
if rel_ids == list(range(len(relationships_final_df))):
    print(f"  ✅ 관계 ID 연속성 확인: 0 ~ {len(relationships_final_df)-1}")
else:
    print(f"  ❌ 관계 ID에 공백 발견")

# 7. 정규화된 데이터 저장
print("\n💾 정규화된 데이터 저장 중...")

# 엔티티 저장 (이미 수정된 degree 포함)
normalized_entities_path = os.path.join(SNU_KG_OUTPUT, "final", "entities_normalized.parquet")
os.makedirs(os.path.dirname(normalized_entities_path), exist_ok=True)
entities_final_df.to_parquet(normalized_entities_path, index=False)
print(f"  ✅ 정규화된 엔티티 저장: {normalized_entities_path}")

# 관계 저장 (수정된 combined_degree 포함)
normalized_relationships_path = os.path.join(SNU_KG_OUTPUT, "final", "relationships_normalized.parquet")
relationships_final_df.to_parquet(normalized_relationships_path, index=False)
print(f"  ✅ 정규화된 관계 저장: {normalized_relationships_path}")

# 8. GraphRAG 형식으로 최종 저장
print("\n📦 GraphRAG 형식으로 최종 저장...")

# GraphRAG output 디렉토리
graphrag_output_dir = os.path.join(GRAPHRAG_WORKSPACE, "output")

# output_after_kggen 디렉토리 생성
output_after_kggen_dir = os.path.join(GRAPHRAG_WORKSPACE, "output_after_kggen")
print(f"\n📁 output_after_kggen 디렉토리 생성 중...")

# 디렉토리가 이미 있으면 삭제하고 새로 생성
if os.path.exists(output_after_kggen_dir):
    shutil.rmtree(output_after_kggen_dir)
    print(f"  - 기존 디렉토리 삭제")

os.makedirs(output_after_kggen_dir, exist_ok=True)
print(f"  ✅ 디렉토리 생성: {output_after_kggen_dir}")

# 1. output/ 디렉토리의 모든 파일을 output_after_kggen/으로 복사
print(f"\n📋 원본 GraphRAG 출력 파일들을 복사 중...")
if os.path.exists(graphrag_output_dir):
    for file in os.listdir(graphrag_output_dir):
        src_path = os.path.join(graphrag_output_dir, file)
        dst_path = os.path.join(output_after_kggen_dir, file)
        
        # 파일인 경우만 복사 (디렉토리는 제외)
        if os.path.isfile(src_path):
            shutil.copy2(src_path, dst_path)  # copy2는 메타데이터도 복사
            print(f"  ✅ {file} 복사됨")
    
    # 복사된 파일 목록
    copied_files = [f for f in os.listdir(output_after_kggen_dir) if os.path.isfile(os.path.join(output_after_kggen_dir, f))]
    print(f"\n  총 {len(copied_files)}개 파일 복사 완료")
else:
    print(f"  ⚠️ 원본 output 디렉토리가 없습니다: {graphrag_output_dir}")

# 2. 정규화된 entities.parquet과 relationships.parquet로 덮어쓰기
print(f"\n🔄 정규화된 파일로 덮어쓰기 중...")
shutil.copy(normalized_entities_path, os.path.join(output_after_kggen_dir, "entities.parquet"))
print(f"  ✅ entities.parquet 덮어쓰기 완료 (kg-gen 정규화 버전)")

shutil.copy(normalized_relationships_path, os.path.join(output_after_kggen_dir, "relationships.parquet"))
print(f"  ✅ relationships.parquet 덮어쓰기 완료 (kg-gen 정규화 버전)")

# 3. 원본 output 디렉토리는 그대로 유지 (백업으로)
print(f"\n📦 최종 디렉토리 구조:")
print(f"  - {graphrag_output_dir} (원본 GraphRAG 출력 - 보존됨)")
print(f"  - {output_after_kggen_dir} (kg-gen 정규화 적용 버전)")

# 최종 파일 확인
print(f"\n📋 output_after_kggen 디렉토리 내용:")
for file in sorted(os.listdir(output_after_kggen_dir)):
    file_path = os.path.join(output_after_kggen_dir, file)
    if os.path.isfile(file_path):
        size_mb = os.path.getsize(file_path) / 1024 / 1024
        
        # entities와 relationships는 정규화됨을 표시
        if file in ["entities.parquet", "relationships.parquet"]:
            print(f"  - {file} ({size_mb:.2f} MB) [kg-gen 정규화됨]")
        else:
            print(f"  - {file} ({size_mb:.2f} MB) [원본]")

print(f"\n✅ 모든 파일이 output_after_kggen에 저장되었습니다.")

# 요약 통계
print("\n" + "="*60)
print("📊 정규화 완료 요약:")
print(f"  - 최종 엔티티 수: {len(entities_final_df)}")
print(f"  - 최종 관계 수: {len(relationships_final_df)}")
print(f"  - 평균 노드 degree: {entities_final_df['degree'].mean():.2f}")
print(f"  - 평균 관계 weight: {relationships_final_df['weight'].mean():.2f}")
print(f"  - 그래프 밀도: {nx.density(G):.4f}")
print(f"  - 엔티티 ID 범위: 0 ~ {len(entities_final_df)-1} (연속)")
print(f"  - 관계 ID 범위: 0 ~ {len(relationships_final_df)-1} (연속)")

print("\n✅ C단계 완료: 정규화 과정 완료")
print("="*60)

## 13. GraphRAG 나머지 워크플로우 실행 (D단계)

kg-gen으로 중복 제거된 엔티티와 관계를 바탕으로 GraphRAG의 나머지 워크플로우(커뮤니티 탐지, 임베딩 생성 등)를 실행합니다.

In [ ]:
GRAPHRAG_WORKSPACE = os.path.join(SNU_KG_OUTPUT, "graphrag_workspace")

In [ ]:
# D. GraphRAG 나머지 워크플로우 실행
print("="*60)
print("D. GraphRAG 나머지 워크플로우 실행 시작")
print("="*60)

# 환경 변수 설정
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print("✅ TOKENIZERS_PARALLELISM 환경 변수 설정 완료")

# output_after_kggen 디렉토리 확인
output_after_kggen_dir = os.path.join(GRAPHRAG_WORKSPACE, "output_after_kggen")
if not os.path.exists(output_after_kggen_dir):
    print(f"❌ output_after_kggen 디렉토리를 찾을 수 없습니다: {output_after_kggen_dir}")
    raise FileNotFoundError(f"Directory not found: {output_after_kggen_dir}")

print(f"✅ output_after_kggen 디렉토리 확인: {output_after_kggen_dir}")

# 현재 작업 디렉토리 확인
current_dir = os.getcwd()
os.chdir(GRAPHRAG_WORKSPACE)
print(f"📁 작업 디렉토리: {GRAPHRAG_WORKSPACE}")

# 기존 settings.yaml 백업
settings_file = os.path.join(GRAPHRAG_WORKSPACE, "settings.yaml")
settings_backup = settings_file + ".d_stage_backup"
if os.path.exists(settings_file):
    shutil.copy(settings_file, settings_backup)
    print(f"📦 기존 settings.yaml 백업: {settings_backup}")

# 원본 settings.yaml 읽기
with open(settings_file, 'r', encoding='utf-8') as f:
    settings = yaml.safe_load(f)

# 원본 스타일로 단순하게 수정
# 1. input/output 경로를 output_after_kggen으로 변경
settings['input']['base_dir'] = 'output_after_kggen'
settings['output']['base_dir'] = 'output_after_kggen'
settings['cache']['base_dir'] = 'cache_after_kggen'
settings['reporting']['base_dir'] = 'logs_after_kggen'

# 2. 벡터 스토어 설정 수정 - 올바른 구조로
if 'vector_store' in settings:
    # vector_store가 딕셔너리 구조인지 확인
    if 'default_vector_store' in settings['vector_store']:
        # 기존 구조 유지하며 경로만 수정
        settings['vector_store']['default_vector_store']['db_uri'] = 'output_after_kggen/lancedb'
    else:
        # 잘못된 구조인 경우 재구성
        settings['vector_store'] = {
            'default_vector_store': {
                'type': 'lancedb',
                'db_uri': 'output_after_kggen/lancedb',
                'container_name': 'default'
            }
        }

# 3. 워크플로우 순서 수정 - 표준 파이프라인 순서에 맞춰서
# GraphRAG의 표준 워크플로우 순서:
# extract_covariates → create_communities → create_final_text_units → create_community_reports → generate_text_embeddings
settings['workflows'] = [
    # "extract_covariates",        # 1. covariates 추출 (커뮤니티 생성 전에)
    # "create_communities",        # 2. 커뮤니티 생성
    # "create_final_text_units",   # 3. text_units 업데이트
    # "create_community_reports",  # 4. 커뮤니티 리포트 생성
    "generate_text_embeddings"   # 5. 임베딩 생성
]

print("\n📋 실행할 워크플로우 (표준 순서):")
for i, wf in enumerate(settings['workflows'], 1):
    print(f"   {i}. {wf}")

# 디버깅을 위한 설정 내용 출력
print("\n🔍 수정된 설정 확인:")
print(f"   - input.base_dir: {settings['input']['base_dir']}")
print(f"   - output.base_dir: {settings['output']['base_dir']}")
print(f"   - vector_store.default_vector_store.db_uri: {settings['vector_store']['default_vector_store']['db_uri']}")

# 수정된 설정 저장
with open(settings_file, 'w', encoding='utf-8') as f:
    yaml.dump(settings, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print("\n✅ settings.yaml 수정 완료")

# GraphRAG 실행
print("\n" + "="*60)
print("🚀 GraphRAG 나머지 워크플로우 실행")
print("="*60)

try:
    # 환경 변수 설정
    env = os.environ.copy()
    env['TOKENIZERS_PARALLELISM'] = 'false'
    
    # graphrag index 실행
    index_cmd = [
        sys.executable, "-m", "graphrag", "index",
        "--root", ".",
        "--verbose"
    ]
    
    print("💻 실행 명령어:")
    print(f"   {' '.join(index_cmd)}")
    
    start_time = time.time()
    
    # 프로세스 실행
    process = subprocess.Popen(
        index_cmd,
        cwd=GRAPHRAG_WORKSPACE,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )
    
    # 실시간 출력 - 더 상세한 디버깅 정보
    print("\n📊 진행 상황:")
    stdout_lines = []
    stderr_lines = []
    current_workflow = None
    
    for line in process.stdout:
        stdout_lines.append(line)
        line_strip = line.strip()
        
        if line_strip:
            # 워크플로우 시작/종료 추적
            if "workflow:" in line_strip.lower() or "executing:" in line_strip.lower():
                for wf in settings['workflows']:
                    if wf in line_strip.lower():
                        current_workflow = wf
                        print(f"\n   🔄 [{current_workflow}] 시작...")
                        break
            
            # 주요 진행 상황 표시
            if any(keyword in line_strip.lower() for keyword in [
                'workflow', 'executing', 'completed', 'creating', 'extracting',
                'loading', 'writing', 'error', 'warning', 'exception',
                'covariates', 'communities', 'embeddings', 'reports'
            ]):
                # 현재 워크플로우 표시
                if current_workflow:
                    print(f"   [{current_workflow}] {line_strip}")
                else:
                    print(f"   → {line_strip}")
            
            # 에러 감지
            if any(err in line_strip.lower() for err in ['error', 'exception', 'failed']):
                print(f"   ⚠️ 오류 감지: {line_strip}")
    
    # stderr 수집
    stderr_output = process.stderr.read()
    if stderr_output:
        stderr_lines = stderr_output.split('\n')
    
    process.wait()
    
    if process.returncode == 0:
        print("\n✅ GraphRAG 워크플로우 실행 성공!")
    else:
        print(f"\n❌ GraphRAG 워크플로우 실행 실패 (Exit code: {process.returncode})")
        
        # 상세한 오류 분석
        print("\n🔍 오류 분석:")
        
        # stderr에서 주요 오류 찾기
        if stderr_lines:
            print("\n📄 stderr 오류 메시지:")
            error_found = False
            for i, line in enumerate(stderr_lines):
                if line.strip():
                    # Python 오류 추적
                    if "Traceback" in line:
                        error_found = True
                        # Traceback 전체 출력
                        print(f"\n   Python 오류 추적:")
                        for j in range(i, min(i + 30, len(stderr_lines))):
                            if stderr_lines[j].strip():
                                print(f"   {stderr_lines[j]}")
                        break
                    # 일반 오류 메시지
                    elif any(err in line.lower() for err in ['error:', 'exception:', 'failed:']):
                        print(f"   ! {line}")
                        error_found = True
            
            if not error_found:
                # 마지막 몇 줄 출력
                print("   (마지막 10줄)")
                for line in stderr_lines[-10:]:
                    if line.strip():
                        print(f"   {line}")
        
        # stdout에서 오류 정보 찾기
        if stdout_lines:
            error_lines = [l for l in stdout_lines if any(err in l.lower() for err in ['error', 'exception', 'failed', 'missing'])]
            if error_lines:
                print("\n📄 stdout에서 발견된 오류:")
                for line in error_lines[-10:]:
                    print(f"   {line.strip()}")
    
    elapsed_time = time.time() - start_time
    print(f"\n⏱️ 총 소요 시간: {elapsed_time//60:.0f}분 {elapsed_time%60:.0f}초")
    
except Exception as e:
    print(f"\n❌ 실행 중 예외 발생: {str(e)}")
    import traceback
    traceback.print_exc()

finally:
    # 원래 디렉토리로 복귀
    os.chdir(current_dir)
    
    # 백업된 settings 복원
    if os.path.exists(settings_backup):
        shutil.copy(settings_backup, settings_file)
        os.remove(settings_backup)
        print(f"\n🔄 원본 settings.yaml 복원 완료")

# 최종 결과 확인
print("\n" + "="*60)
print("📊 최종 결과 확인:")
print("="*60)

output_files = [
    ("entities.parquet", "파일"),
    ("relationships.parquet", "파일"),
    ("communities.parquet", "파일"),
    ("community_reports.parquet", "파일"),
    ("covariates.parquet", "파일"),  # 두 가지 이름으로 저장될 수 있음
    ("text_units.parquet", "파일"),  # 업데이트된 text_units
    ("lancedb", "디렉토리")
]

success_count = 0
generated_files = []

for file_name, file_type in output_files:
    file_path = os.path.join(output_after_kggen_dir, file_name)
    if os.path.exists(file_path):
        if file_type == "파일" and os.path.isfile(file_path):
            size_mb = os.path.getsize(file_path) / 1024 / 1024
            print(f"  ✅ {file_name} ({size_mb:.2f} MB)")
            generated_files.append(file_name)
            if file_name.endswith('.parquet'):
                try:
                    df = pd.read_parquet(file_path)
                    print(f"     → {len(df)} rows")
                    if file_name == "communities.parquet" and 'level' in df.columns:
                        level_counts = df['level'].value_counts().sort_index()
                        print(f"     → 레벨 분포: {dict(level_counts)}")
                except Exception as e:
                    print(f"     → 읽기 오류: {str(e)[:50]}")
            success_count += 1
        elif file_type == "디렉토리" and os.path.isdir(file_path):
            print(f"  ✅ {file_name}/ (디렉토리)")
            # LanceDB 테이블 확인
            try:
                tables = os.listdir(file_path)
                print(f"     → 테이블: {', '.join([t for t in tables if not t.startswith('.')])}")
            except:
                pass
            generated_files.append(file_name)
            success_count += 1
    else:
        print(f"  ❌ {file_name} - 생성되지 않음")

# 최종 통계
print("\n" + "="*60)
if success_count >= 7:  # 주요 파일들이 대부분 생성되었다면
    print("🎉 SNU-KG 파이프라인 성공적으로 완료!")
else:
    print(f"⚠️ SNU-KG 파이프라인 부분적으로 완료 ({success_count}/{len(output_files)} 파일 생성)")

print(f"\n📋 생성된 파일:")
for f in generated_files:
    print(f"   • {f}")
    
# print(f"\n📊 처리 결과:")
# print(f"   • 원본 엔티티: {len(entities_df)} → kg-gen 후: {len(entities_final_df)} ({((len(entities_df)-len(entities_final_df))/len(entities_df)*100):.1f}% 감소)")
# print(f"   • 원본 관계: {len(relationships_df)} → kg-gen 후: {len(relationships_final_df)} ({((len(relationships_df)-len(relationships_final_df))/len(relationships_df)*100):.1f}% 감소)")

# print(f"\n📂 모든 결과가 저장된 위치:")
# print(f"   • 최종 GraphRAG 출력: {output_after_kggen_dir}")
# print(f"   • 원본 GraphRAG 출력 (보존): {os.path.join(GRAPHRAG_WORKSPACE, 'output')}")
# print(f"   • SNU-KG 중간 파일: {os.path.join(SNU_KG_OUTPUT, 'intermediate')}")
# print(f"   • SNU-KG 정규화 파일: {os.path.join(SNU_KG_OUTPUT, 'final')}")

print("\n✅ D단계 완료: GraphRAG 나머지 워크플로우 실행 완료")
print("="*60)

## 14. 원본 데이터에 대해 GraphRAG 나머지 워크플로우 실행

kg-gen으로 정규화하지 않은 원본 GraphRAG 출력에 대해서도 동일한 나머지 워크플로우를 실행하여 비교 분석할 수 있도록 합니다.

In [ ]:
print("\n" + "="*80)
print("14. 원본 데이터에 대해 GraphRAG 나머지 워크플로우 실행")
print("="*80 + "\n")

# 환경 변수 설정
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print("✅ TOKENIZERS_PARALLELISM 환경 변수 설정 완료")

# 원본 데이터 경로 설정
original_output_dir = os.path.join(GRAPHRAG_WORKSPACE, "output")
print(f"📁 원본 데이터 경로: {original_output_dir}\n")

# 현재 작업 디렉토리 저장
original_working_dir = os.getcwd()
os.chdir(GRAPHRAG_WORKSPACE)
print(f"📁 작업 디렉토리: {GRAPHRAG_WORKSPACE}")

# 기존 settings.yaml 백업
settings_file = os.path.join(GRAPHRAG_WORKSPACE, "settings.yaml")
settings_backup_original = settings_file + ".original_stage_backup"
if os.path.exists(settings_file):
    shutil.copy(settings_file, settings_backup_original)
    print(f"📦 기존 settings.yaml 백업: {settings_backup_original}")

# 원본 settings.yaml 읽기
with open(settings_file, 'r', encoding='utf-8') as f:
    settings = yaml.safe_load(f)

# 원본 데이터용 설정 수정
# 1. input/output 경로를 원본 output으로 설정
settings['input']['base_dir'] = 'output'  # 원본 데이터 경로
settings['output']['base_dir'] = 'output'
settings['cache']['base_dir'] = 'cache'
settings['reporting']['base_dir'] = 'logs'

# 2. 벡터 스토어 설정
if 'vector_store' in settings:
    if 'default_vector_store' in settings['vector_store']:
        settings['vector_store']['default_vector_store']['db_uri'] = 'output/lancedb'
    else:
        settings['vector_store'] = {
            'default_vector_store': {
                'type': 'lancedb',
                'db_uri': 'output/lancedb',
                'container_name': 'default'
            }
        }

# 3. extract_claims 활성화
if 'extract_claims' not in settings:
    settings['extract_claims'] = {}
settings['extract_claims']['enabled'] = True
settings['extract_claims']['model_id'] = 'default_chat_model'
settings['extract_claims']['description'] = "한국 농업 관련 주요 주장, 사실, 통계 등을 추출합니다."

# 4. 워크플로우 설정 - GraphRAG 표준 순서
settings['workflows'] = [
    # "extract_covariates",        # 1. covariates 추출
    # "create_communities",        # 2. 커뮤니티 생성
    # "create_final_text_units",   # 3. text_units 업데이트
    # "create_community_reports",  # 4. 커뮤니티 리포트 생성
    "generate_text_embeddings"   # 5. 임베딩 생성
]

print("\n📋 원본 데이터에 대해 실행할 워크플로우:")
for i, wf in enumerate(settings['workflows'], 1):
    print(f"   {i}. {wf}")

# 설정 확인
print("\n🔍 원본 데이터용 설정 확인:")
print(f"   - input.base_dir: {settings['input']['base_dir']}")
print(f"   - output.base_dir: {settings['output']['base_dir']}")
print(f"   - vector_store.default_vector_store.db_uri: {settings['vector_store']['default_vector_store']['db_uri']}")

# 수정된 설정 저장
with open(settings_file, 'w', encoding='utf-8') as f:
    yaml.dump(settings, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print("\n✅ 원본 데이터용 settings.yaml 수정 완료")

# 원본 데이터 파일 확인
print("\n📊 원본 데이터 파일 확인:")
for file in ['entities.parquet', 'relationships.parquet', 'text_units.parquet']:
    file_path = os.path.join(original_output_dir, file)
    if os.path.exists(file_path):
        df = pd.read_parquet(file_path)
        print(f"  ✅ {file}: {len(df)} 행")
    else:
        print(f"  ❌ {file}: 파일 없음")

# GraphRAG 실행
print("\n" + "="*60)
print("🚀 원본 데이터에 대한 GraphRAG 워크플로우 실행")
print("="*60)

try:
    # 환경 변수 설정
    env = os.environ.copy()
    env['TOKENIZERS_PARALLELISM'] = 'false'
    
    # graphrag index 실행
    index_cmd = [
        sys.executable, "-m", "graphrag", "index",
        "--root", ".",
        "--verbose"
    ]
    
    print("💻 실행 명령어:")
    print(f"   {' '.join(index_cmd)}")
    
    start_time = time.time()
    
    # 프로세스 실행
    process = subprocess.Popen(
        index_cmd,
        cwd=GRAPHRAG_WORKSPACE,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1
    )
    
    # 진행 상황 출력
    print("\n📊 진행 상황:")
    current_workflow = None
    
    for line in process.stdout:
        line_strip = line.strip()
        
        if line_strip:
            # 워크플로우 시작/종료 추적
            if "workflow:" in line_strip.lower() or "executing:" in line_strip.lower():
                for wf in settings['workflows']:
                    if wf in line_strip.lower():
                        current_workflow = wf
                        print(f"\n   🔄 [{current_workflow}] 시작...")
                        break
            
            # 주요 진행 상황 표시
            if any(keyword in line_strip.lower() for keyword in [
                'workflow', 'executing', 'completed', 'creating',
                'loading', 'writing', 'error', 'warning'
            ]):
                if current_workflow:
                    print(f"   [{current_workflow}] {line_strip}")
                else:
                    print(f"   → {line_strip}")
    
    process.wait()
    
    if process.returncode == 0:
        print("\n✅ 원본 데이터 워크플로우 실행 성공!")
    else:
        print(f"\n❌ 원본 데이터 워크플로우 실행 실패 (Exit code: {process.returncode})")
    
    elapsed_time = time.time() - start_time
    print(f"\n⏱️ 총 소요 시간: {elapsed_time//60:.0f}분 {elapsed_time%60:.0f}초")
    
except Exception as e:
    print(f"\n❌ 실행 중 예외 발생: {str(e)}")
    import traceback
    traceback.print_exc()

finally:
    # 원래 디렉토리로 복귀
    os.chdir(original_working_dir)
    
    # 백업된 settings 복원
    if os.path.exists(settings_backup_original):
        shutil.copy(settings_backup_original, settings_file)
        os.remove(settings_backup_original)
        print(f"\n🔄 원본 settings.yaml 복원 완료")

# 결과 파일 확인
print("\n📊 생성된 파일 확인:")
expected_files = [
    'covariates.parquet',
    'communities.parquet',
    'community_reports.parquet',
    'text_units.parquet',
    'lancedb'
]

for file in expected_files:
    file_path = os.path.join(original_output_dir, file)
    if os.path.exists(file_path):
        if file.endswith('.parquet'):
            df = pd.read_parquet(file_path)
            print(f"  ✅ {file}: {len(df)} 행")
            if file == 'communities.parquet' and 'level' in df.columns:
                level_counts = df['level'].value_counts().sort_index()
                print(f"     → 레벨 분포: {dict(level_counts)}")
        else:
            print(f"  ✅ {file}/: 디렉토리 생성됨")
    else:
        print(f"  ❌ {file}: 생성되지 않음")

# 원본과 kg-gen 정규화 버전 비교
print("\n" + "="*60)
print("📊 원본 vs kg-gen 정규화 버전 비교")
print("="*60)

# 엔티티 비교
print("\n📌 엔티티 수:")
original_entities = pd.read_parquet(os.path.join(original_output_dir, 'entities.parquet'))
kggen_entities = pd.read_parquet(os.path.join(output_after_kggen_dir, 'entities.parquet'))
print(f"  - 원본: {len(original_entities)}개")
print(f"  - kg-gen 정규화: {len(kggen_entities)}개")
print(f"  - 감소율: {((len(original_entities) - len(kggen_entities)) / len(original_entities) * 100):.1f}%")

# 관계 비교
print("\n📌 관계 수:")
original_relations = pd.read_parquet(os.path.join(original_output_dir, 'relationships.parquet'))
kggen_relations = pd.read_parquet(os.path.join(output_after_kggen_dir, 'relationships.parquet'))
print(f"  - 원본: {len(original_relations)}개")
print(f"  - kg-gen 정규화: {len(kggen_relations)}개")
print(f"  - 감소율: {((len(original_relations) - len(kggen_relations)) / len(original_relations) * 100):.1f}%")

# 커뮤니티 비교
if os.path.exists(os.path.join(original_output_dir, 'communities.parquet')) and \
   os.path.exists(os.path.join(output_after_kggen_dir, 'communities.parquet')):
    print("\n📌 커뮤니티 수:")
    original_communities = pd.read_parquet(os.path.join(original_output_dir, 'communities.parquet'))
    kggen_communities = pd.read_parquet(os.path.join(output_after_kggen_dir, 'communities.parquet'))
    print(f"  - 원본: {len(original_communities)}개")
    print(f"  - kg-gen 정규화: {len(kggen_communities)}개")
    print(f"  - 차이: {abs(len(kggen_communities) - len(original_communities))}개")
    
    # 레벨별 비교
    if 'level' in original_communities.columns and 'level' in kggen_communities.columns:
        print("\n  레벨별 커뮤니티 분포:")
        orig_levels = original_communities['level'].value_counts().sort_index()
        kggen_levels = kggen_communities['level'].value_counts().sort_index()
        
        all_levels = set(orig_levels.index) | set(kggen_levels.index)
        for level in sorted(all_levels):
            orig_count = orig_levels.get(level, 0)
            kggen_count = kggen_levels.get(level, 0)
            print(f"    Level {level}: 원본 {orig_count}개 → kg-gen {kggen_count}개")

# Covariates 비교
if os.path.exists(os.path.join(original_output_dir, 'covariates.parquet')) and \
   os.path.exists(os.path.join(output_after_kggen_dir, 'covariates.parquet')):
    print("\n📌 Covariates 수:")
    original_covariates = pd.read_parquet(os.path.join(original_output_dir, 'covariates.parquet'))
    kggen_covariates = pd.read_parquet(os.path.join(output_after_kggen_dir, 'covariates.parquet'))
    print(f"  - 원본: {len(original_covariates)}개")
    print(f"  - kg-gen 정규화: {len(kggen_covariates)}개")

print("\n" + "="*60)
print("✅ 14단계 완료: 원본 데이터 워크플로우 실행 및 비교 분석 완료")
print("="*60)